# Week 8: Accessible web front-end (Colab version)

Self-contained: choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
It writes the project code and the web page, collects the 38 official pages live, runs the tests, times live answers on the GPU (task T35a) and opens the web app in a new browser tab through Colab.

Downloads four models (about 15 GB). Pages are fetched live, so answers can differ slightly from the local run.

**Ethics note:** in Colab the questions are processed on a Google virtual machine. Use this for development and timing only, not with study participants unless they have been told (`docs/ethics-notes.md`).

In [ ]:
%pip install -q -U transformers sentence-transformers faiss-cpu rank-bm25 lingua-language-detector sentencepiece

In [ ]:
import os, sys
for d in ['src', 'eval', 'web', 'tests', 'data/raw', 'data/processed', 'results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/sources.py
"""Curated list of official sources for the three domains.

Every entry was checked against the live site on 28 Oct 2026 (HTTP 200).
GOV.UK pages are fetched through the public Content API; NHS pages as HTML.
Both are published under the Open Government Licence v3.0, and NHS reuse
also needs attribution, a link back and a date notice (nhs.uk terms).
"""

GOVUK = {
    "schools": [
        "schools-admissions",
        "apply-for-primary-school-place",
        "apply-for-secondary-school-place",
        "school-attendance-absence",
        "apply-free-school-meals",
        "children-with-special-educational-needs",
        "free-school-transport",
        "school-uniform",
        "types-of-school",
        "school-term-holiday-dates",
        "know-when-you-can-leave-school",
        "complain-about-school",
    ],
    "housing": [
        "private-renting",
        "private-renting-evictions",
        "guidance/renters-rights-act-overview-for-tenants",
        "tenancy-deposit-protection",
        "council-housing",
        "housing-association-homes",
        "council-housing-association-evictions",
        "housing-benefit",
        "universal-credit",
        "homelessness-help-from-council",
        "council-tax",
        "apply-council-tax-reduction",
    ],
    "health": [
        "help-nhs-costs",
        "healthcare-immigration-application",
    ],
}

NHS = {
    "health": [
        "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
        "https://www.nhs.uk/nhs-services/gps/gp-appointments-and-bookings/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-go-to-ae/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-call-999/",
        "https://www.nhs.uk/nhs-services/dentists/how-to-find-an-nhs-dentist/",
        "https://www.nhs.uk/nhs-services/dentists/how-much-nhs-dental-treatment-costs/",
        "https://www.nhs.uk/nhs-services/pharmacies/how-pharmacies-can-help/",
        "https://www.nhs.uk/nhs-services/help-with-health-costs/nhs-low-income-scheme-lis/",
        "https://www.nhs.uk/nhs-services/visiting-or-moving-to-england/how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/",
        "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/",
        "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/",
    ],
}


In [ ]:
%%writefile src/ingest.py
"""Fetch every source page, keep a raw snapshot, and write one clean record per page.

Output: data/raw/<id>.(json|html) and data/processed/pages.jsonl
Each record keeps the section structure (h2/h3 headings) so chunks can carry
their heading, plus three dates: the publisher's own date, the publisher's
last technical update, and when we fetched it (for data-currency reporting).
"""
import hashlib
import json
import pathlib
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

from sources import GOVUK, NHS

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "pages.jsonl"
UA = {"User-Agent": "MSc research prototype (multilingual civic assistant); polite crawler"}

# GOV.UK formats store text in different fields; order follows the rendered page.
GOVUK_TEXT_FIELDS = ["introduction", "introductory_paragraph", "body", "more_information",
                     "need_to_know", "other_ways_to_apply", "what_you_need_to_know"]


def page_id(url):
    return hashlib.sha1(url.encode()).hexdigest()[:10]


def sections_from_html(html, page_title):
    """Split HTML into (heading, text) sections at h2/h3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    sections, heading, buf = [], page_title, []
    for el in soup.find_all(["h2", "h3", "p", "li", "td", "th"]):
        if el.name in ("h2", "h3"):
            if buf:
                sections.append({"heading": heading, "text": " ".join(buf)})
            heading, buf = el.get_text(" ", strip=True), []
        elif not el.find_parent(["li", "td", "th"]) or el.name in ("td", "th"):
            # skip nested elements already covered by their parent's text
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    if buf:
        sections.append({"heading": heading, "text": " ".join(buf)})
    return sections


def fetch_govuk(path, domain):
    r = requests.get("https://www.gov.uk/api/content/" + path, headers=UA, timeout=30)
    r.raise_for_status()
    data = r.json()
    url = "https://www.gov.uk/" + path
    (RAW / f"{page_id(url)}.json").write_text(r.text)
    det = data.get("details", {})
    html = "".join(det.get(f, "") for f in GOVUK_TEXT_FIELDS if isinstance(det.get(f), str))
    for part in det.get("parts", []):
        html += f"<h2>{part['title']}</h2>{part['body']}"
    title = data.get("title", "")
    return {
        "id": page_id(url), "url": url, "source": "GOV.UK", "domain": domain,
        "title": title, "format": data.get("schema_name"),
        "public_updated_at": (data.get("public_updated_at") or "")[:10],
        "updated_at": (data.get("updated_at") or "")[:10],
        "sections": sections_from_html(html, title),
    }


def fetch_nhs(url, domain):
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    (RAW / f"{page_id(url)}.html").write_text(r.text)
    soup = BeautifulSoup(r.text, "html.parser")
    reviewed = soup.find(string=lambda s: s and "Page last reviewed" in s)
    reviewed = reviewed.strip().split(":", 1)[1].strip() if reviewed else ""
    modified = ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            modified = (json.loads(s.string or "{}").get("dateModified") or modified)[:10]
        except (ValueError, AttributeError):
            pass
    article = soup.find("article") or soup.find("main")
    for junk in article.find_all(["aside", "nav", "footer", "script", "style"]):
        junk.decompose()
    title = soup.find("h1").get_text(" ", strip=True)
    # the lead sentence ("999 is for life-threatening emergencies...") sits outside <article>
    lede = soup.find(class_="nhsuk-lede-text")
    html = (str(lede) if lede and article not in lede.parents else "") + str(article)
    return {
        "id": page_id(url), "url": url, "source": "NHS.uk", "domain": domain,
        "title": title, "format": "nhs_article",
        "public_updated_at": datetime.strptime(reviewed, "%d %B %Y").strftime("%Y-%m-%d") if reviewed else "",
        "updated_at": modified,
        "sections": sections_from_html(html, title),
    }


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    jobs = [(fetch_govuk, p, d) for d, ps in GOVUK.items() for p in ps]
    jobs += [(fetch_nhs, u, d) for d, us in NHS.items() for u in us]
    records, failed = [], []
    for fn, target, domain in jobs:
        try:
            rec = fn(target, domain)
            rec["fetched_at"] = fetched_at
            rec["words"] = sum(len(s["text"].split()) for s in rec["sections"])
            records.append(rec)
            print(f"ok   {rec['domain']:8} {rec['words']:5}w  {rec['title']}")
        except Exception as e:  # keep going; report at the end
            failed.append((target, repr(e)))
            print(f"FAIL {target}: {e}")
        time.sleep(0.5)
    with OUT.open("w") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"\n{len(records)} pages written to {OUT.relative_to(ROOT)}; {len(failed)} failed")
    for t, e in failed:
        print("  failed:", t, e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/chunk.py
"""Turn pages into retrievable chunks that respect the page's own sections.

Rules (kept simple so they are easy to describe in the methodology):
  * one chunk per h2/h3 section;
  * each section is tagged with the UK nation it applies to (from its heading);
  * sections shorter than MIN_WORDS are merged into the next one on the same page,
    but only when both have the same nation tag;
  * sections longer than MAX_WORDS are split into windows with OVERLAP words;
  * every chunk text starts with "<page title> - <section heading>:" so the
    embedding sees the context a reader would see.
Output: data/processed/chunks.csv
"""
import json
import pathlib
import re

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
MIN_WORDS, MAX_WORDS, OVERLAP = 40, 200, 40
OTHER_NATIONS = ["Scotland", "Wales", "Northern Ireland"]


def nation_of(heading):
    """GOV.UK marks nation-specific sections in the heading ("Scotland", "If you're in Wales...").
    Everything else applies to England (or the whole UK). NHS.uk pages cover England only."""
    named = [n for n in OTHER_NATIONS if re.search(rf"\b{n}\b", heading)]
    return ";".join(n.lower().replace(" ", "_") for n in named) or "england"


def join(a, b):
    return {"heading": a["heading"] + " / " + b["heading"], "text": a["text"] + " " + b["text"],
            "nation": a["nation"]}


def merge_small(sections):
    merged, carry = [], None
    for s in sections:
        s = {**s, "nation": nation_of(s["heading"])}
        if carry and carry["nation"] == s["nation"]:
            s = join(carry, s)
        elif carry:
            merged.append(carry)  # never merge across nations; keep the short one alone
        carry = None
        if len(s["text"].split()) < MIN_WORDS:
            carry = s
        else:
            merged.append(s)
    if carry:  # trailing short section joins the previous one if same nation, or stands alone
        if merged and merged[-1]["nation"] == carry["nation"]:
            merged[-1] = join(merged[-1], carry)
        else:
            merged.append(carry)
    return merged


def windows(words):
    if len(words) <= MAX_WORDS:
        return [words]
    step = MAX_WORDS - OVERLAP
    return [words[i:i + MAX_WORDS] for i in range(0, len(words) - OVERLAP, step)]


def main():
    pages = [json.loads(l) for l in open(ROOT / "data/processed/pages.jsonl")]
    rows = []
    for p in pages:
        for s_i, s in enumerate(merge_small(p["sections"])):
            for w_i, w in enumerate(windows(s["text"].split())):
                body = " ".join(w)
                rows.append({
                    "chunk_id": f"{p['id']}-{s_i:02d}-{w_i}",
                    "page_id": p["id"], "url": p["url"], "source": p["source"], "domain": p["domain"],
                    "title": p["title"], "heading": s["heading"], "nation": s["nation"],
                    "public_updated_at": p["public_updated_at"], "fetched_at": p["fetched_at"],
                    "words": len(w), "body": body,
                    "text": f"{p['title']} - {s['heading']}: {body}",
                })
    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data/processed/chunks.csv", index=False)
    print(f"{len(pages)} pages -> {len(df)} chunks")
    print(df.groupby("domain").agg(pages=("page_id", "nunique"), chunks=("chunk_id", "count"),
                                   words=("words", "sum")).to_string())
    print("nation tags:", df.nation.value_counts().to_dict())
    print("chunk words: median", int(df.words.median()), "min", df.words.min(), "max", df.words.max())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieval.py
"""Dense and lexical retrievers over the chunk table.

Dense: sentence-transformers model + FAISS inner-product index on normalised
vectors (= cosine similarity). E5 models need "query: " / "passage: " prefixes.
Lexical: BM25 (rank_bm25), used as a non-neural baseline.
"""
import re

import torch  # noqa: F401  must load before faiss: on macOS the two OpenMP runtimes otherwise crash NLLB generation
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class DenseRetriever:
    def __init__(self, model_name, texts):
        self.model = SentenceTransformer(model_name)
        self.e5 = "e5" in model_name.lower()
        passages = [("passage: " + t) if self.e5 else t for t in texts]
        emb = self.model.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, k=10):
        q = self.model.encode([("query: " + query) if self.e5 else query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def save(self, path):
        faiss.write_index(self.index, str(path))


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, texts):
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        ids = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in ids]


In [ ]:
%%writefile src/language.py
"""Detect which supported language a question is written in.

Uses lingua (Apache-2.0), restricted to the five project languages, which makes
short questions much easier to classify than open-set detection. Arabic and Urdu
share a script, so restricting the set matters most for those two.
"""
from lingua import Language, LanguageDetectorBuilder

SUPPORTED = {
    Language.ENGLISH: "en", Language.POLISH: "pl", Language.ROMANIAN: "ro",
    Language.ARABIC: "ar", Language.URDU: "ur",
}
_detector = LanguageDetectorBuilder.from_languages(*SUPPORTED).build()


def detect(text):
    """Return 'en', 'pl', 'ro', 'ar' or 'ur'; falls back to 'en' if lingua cannot decide."""
    lang = _detector.detect_language_of(text)
    return SUPPORTED.get(lang, "en")


In [ ]:
%%writefile src/translate.py
"""Machine translation with NLLB-200 (facebook/nllb-200-distilled-600M, CC-BY-NC-4.0).

Used in week 5 to translate questions into English before retrieval, and in week 6
for answers going back into the user's language. Non-commercial licence: fine for
this research prototype, not for a production service.
"""
import re

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
NLLB_CODES = {"en": "eng_Latn", "pl": "pol_Latn", "ro": "ron_Latn", "ar": "arb_Arab", "ur": "urd_Arab"}


class Translator:
    def __init__(self, model_name=MODEL, device=None):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        # Default as in the week 5-7 experiments: GPU on Colab, CPU elsewhere. The web app (week 8) passes
        # "mps" on Apple silicon, where CPU translation can take minutes when other programs are busy.
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).eval().to(self.device)

    @torch.inference_mode()
    def _translate_sentences(self, sentences, src, tgt, max_length):
        self.tok.src_lang = NLLB_CODES[src]
        batch = self.tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=400).to(self.device)
        out = self.model.generate(
            **batch, max_length=max_length, num_beams=4,
            forced_bos_token_id=self.tok.convert_tokens_to_ids(NLLB_CODES[tgt]))
        return self.tok.batch_decode(out, skip_special_tokens=True)

    def translate(self, texts, src, tgt="en", max_length=256):
        """Translate a list of strings from language `src` to `tgt` (project codes, e.g. 'pl').

        NLLB is trained on single sentences and silently drops later sentences of a
        multi-sentence input, so each text is split into sentences first and rejoined.
        """
        if src == tgt:
            return list(texts)
        split = [split_sentences(t) for t in texts]
        flat = [s for parts in split for s in parts]
        done = iter(self._translate_sentences(flat, src, tgt, max_length))
        return [" ".join(next(done) for _ in parts) for parts in split]


def split_sentences(text):
    """Split on sentence-final punctuation, including Arabic ؟ and Urdu ۔ ."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?؟۔])\s+", text.strip())]
    return [p for p in parts if p] or [text]


In [ ]:
%%writefile src/rerank.py
"""Second-stage reranking with a multilingual cross-encoder (BAAI/bge-reranker-v2-m3, Apache-2.0).

The bi-encoder (e5) compares a question vector with pre-computed chunk vectors, which
is fast but coarse. A cross-encoder reads the question and one chunk together and
scores how well that chunk answers it; it is too slow for the whole corpus, so it
only reorders the top candidates from the first stage.
"""
from sentence_transformers import CrossEncoder

MODEL = "BAAI/bge-reranker-v2-m3"


class Reranker:
    def __init__(self, model_name=MODEL):
        self.model = CrossEncoder(model_name, max_length=512)

    def rerank(self, query, candidates, texts):
        """candidates: list of chunk ids; texts: chunk id -> text. Returns [(id, score)] best first."""
        if not candidates:
            return []
        scores = self.model.predict([(query, texts[i]) for i in candidates], batch_size=16, show_progress_bar=False)
        return sorted(zip(candidates, map(float, scores)), key=lambda x: -x[1])


In [ ]:
%%writefile src/pipeline.py
"""Week 5 search pipeline: question in any supported language -> best official passages.

    question
      -> detect language                       (language.py)
      -> translate to English if not English   (translate.py, optional)
      -> dense retrieval, original + English   (retrieval.py, e5 + FAISS)
      -> fuse the two ranked lists (RRF)
      -> keep England / UK-wide chunks only     (nation tag from chunk.py)
      -> rerank top candidates                  (rerank.py, optional)
      -> top k passages with title, heading, URL and dates for citation
"""
import pathlib

import pandas as pd

from language import detect
from retrieval import DenseRetriever

ROOT = pathlib.Path(__file__).resolve().parents[1]
E5 = "intfloat/multilingual-e5-base"
RRF_K = 60  # standard constant from Cormack et al. (2009)


def rrf(*ranked_lists, k=RRF_K):
    """Reciprocal rank fusion: combine ranked lists of ids into one ranking."""
    score = {}
    for ranked in ranked_lists:
        for rank, i in enumerate(ranked, start=1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda i: -score[i])


class SearchPipeline:
    def __init__(self, chunks=None, translate=True, rerank=True, england_only=True, pool=20, translator_device=None):
        self.chunks = chunks if chunks is not None else pd.read_csv(ROOT / "data/processed/chunks.csv")
        self.texts = self.chunks.text.tolist()
        self.retriever = DenseRetriever(E5, self.texts)
        self.allowed = set(self.chunks.index[self.chunks.nation == "england"]) if england_only else None
        self.pool = pool
        self.translator = None
        if translate:
            from translate import Translator
            self.translator = Translator(device=translator_device)
        self.reranker = None
        if rerank:
            from rerank import Reranker
            self.reranker = Reranker()

    def _dense(self, query, n):
        return [i for i, _ in self.retriever.search(query, k=n)]

    def search(self, question, k=3, lang=None):
        lang = lang or detect(question)
        english = question
        lists = [self._dense(question, 50)]
        if self.translator and lang != "en":
            english = self.translator.translate([question], src=lang)[0]
            lists.append(self._dense(english, 50))
        ranked = rrf(*lists) if len(lists) > 1 else lists[0]
        if self.allowed is not None:
            ranked = [i for i in ranked if i in self.allowed]
        ranked = ranked[:self.pool]
        scores = {}
        if self.reranker:
            scores = dict(self.reranker.rerank(question, ranked, self.texts))
            ranked = sorted(ranked, key=lambda i: -scores[i])
        results = []
        for i in ranked[:k]:
            c = self.chunks.iloc[i]
            results.append({"chunk": int(i), "title": c.title, "heading": c.heading, "url": c.url,
                            "source": c.source, "nation": c.nation, "public_updated_at": c.public_updated_at,
                            "fetched_at": c.fetched_at, "text": c.body,
                            "score": scores.get(i)})  # reranker relevance (0-1), used by the refusal rule
        return {"lang": lang, "english_query": english, "results": results}


In [ ]:
%%writefile src/generate.py
"""Grounded answer generation with an openly licensed LLM (Qwen/Qwen3.5-4B, Apache-2.0).

Why this model (task T26):
  * Apache-2.0 and not gated, so it can be run locally or on a free Colab GPU and user questions
    never leave the machine (see docs/ethics-notes.md: no third-party API without consent).
  * The model card states support for 201 languages and dialects; Polish, Romanian, Arabic and Urdu
    are all needed here.
  * 4B parameters (9.3 GB of weights in 16-bit): fits a 16 GB Colab T4 in float16 and the 24 GB development laptop in bfloat16.
Qwen3.5 "thinks" before answering by default; thinking is switched off (enable_thinking=False) so the
output is the answer only. Decoding is greedy so that every run gives the same answer.
"""
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3.5-4B"
NO_ANSWER = "NO_ANSWER"
LANGUAGE_NAMES = {"en": "English", "pl": "Polish", "ro": "Romanian", "ar": "Arabic", "ur": "Urdu"}

SYSTEM = """You help people in England understand official guidance on health services, housing and schools.
You are given numbered extracts from official GOV.UK and NHS.uk pages, and a question.

Rules:
1. Use ONLY facts stated in the extracts. Never add facts, dates, places or general remarks from your own knowledge.
2. Every sentence must end with the number of the extract it comes from, in square brackets, for example [1] or [1][2].
   If you cannot point to an extract for a sentence, leave that sentence out.
3. Answer the question first. Write in plain language: short sentences, everyday words, no more than 120 words.
4. If the extracts do not answer the question, reply with exactly NO_ANSWER and nothing else.
5. Do not give medical diagnoses or legal advice about the user's own case; explain what the guidance says.
6. Write the whole answer in {language}."""

# Ungrounded baseline (task T30): same model, same audience and style rules, but no extracts and no citations.
BASELINE = """You help people in England understand official guidance on health services, housing and schools.

Rules:
1. Answer the question first. Write in plain language: short sentences, everyday words, no more than 120 words.
2. If you do not know the answer, say so.
3. Do not give medical diagnoses or legal advice about the user's own case; explain what the guidance says.
4. Write the whole answer in {language}."""

CHECK = """You check whether numbered extracts from official GOV.UK and NHS.uk pages answer a question.
The question may be in any language. Reply YES if the extracts contain the information needed to answer it,
even partly. Reply NO if they do not. Reply with one word: YES or NO."""

CITATION = re.compile(r"\[(\d+)\]")


def format_sources(passages):
    """Number the passages [1]..[k] the way the model will cite them."""
    return "\n\n".join(f"[{n}] {p['title']} - {p['heading']}\n{p['text']}" for n, p in enumerate(passages, 1))


def build_messages(question, passages, answer_lang, pivot=False):
    """pivot=True: the answer is written in English for a question in another language (translate mode).
    The model tends to follow the question's language, so the instruction is repeated after the question."""
    system = SYSTEM.format(language=LANGUAGE_NAMES[answer_lang])
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}"
    if pivot:
        user += "\n\nWrite the answer in English, even though the question is in another language."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def check_messages(question, passages):
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}\n\nDo the extracts answer the question?"
    return [{"role": "system", "content": CHECK}, {"role": "user", "content": user}]


def baseline_messages(question, answer_lang):
    return [{"role": "system", "content": BASELINE.format(language=LANGUAGE_NAMES[answer_lang])},
            {"role": "user", "content": question}]


def citations(answer):
    """Citation numbers used in an answer, in order of first use."""
    return list(dict.fromkeys(int(n) for n in CITATION.findall(answer)))


class Generator:
    def __init__(self, model_name=MODEL, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.bfloat16 if device == "mps" else torch.float32
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device).eval()

    @torch.inference_mode()
    def chat(self, messages, max_new_tokens=300):
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                    enable_thinking=False)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def answerable(self, question, passages):
        """Refusal rule 2: a separate yes/no call - do the passages answer the question?"""
        return self.chat(check_messages(question, passages), max_new_tokens=3).strip().upper().startswith("YES")

    def answer_ungrounded(self, question, answer_lang):
        """Baseline: the same model answers from its own knowledge, with no retrieved sources."""
        return self.chat(baseline_messages(question, answer_lang))

    def answer(self, question, passages, answer_lang, pivot=False):
        """Answer from the passages only. Returns the raw model text (may be NO_ANSWER)."""
        return self.chat(build_messages(question, passages, answer_lang, pivot))


In [ ]:
%%writefile src/assistant.py
"""Week 6 assistant: question in any supported language -> cited answer from official pages, or a refusal.

    question
      -> search stage (pipeline.py, set-up F from week 5): top 3 passages + reranker score
      -> refusal rule 1: top reranker score below THRESHOLD -> refuse without calling the LLM
      -> refusal rule 2: a separate yes/no LLM call says the passages do not answer it -> refuse
      -> LLM answers only from the 3 passages, citing [n]          (generate.py)
      -> refusal rule 3: the LLM replies NO_ANSWER -> refuse
      -> answer language: "direct"    = the LLM writes in the user's language
                          "translate" = the LLM writes in English, NLLB translates it
      -> source cards for the cited passages (title, section, link, date fetched)
      -> health note (NHS 111 / 999) whenever an NHS.uk page is cited

Citations are numbers chosen by the LLM; titles, links and dates always come from the corpus, so the
model can never invent a URL.
"""
from generate import NO_ANSWER, Generator, citations, CITATION
from pipeline import SearchPipeline
from translate import split_sentences

# Calibrated in eval/evaluate_week6.py on 150 answerable and 50 unanswerable questions.
THRESHOLD = 0.0052  # 0.9 x lowest top score of an answerable question whose answer was retrieved

# Fixed messages (written by the author; part of the native-speaker check, task T22).
REFUSAL = {
    "en": "I could not find an answer to this in the official GOV.UK and NHS pages I use. "
          "Please check GOV.UK or NHS.uk, or contact your local council.",
    "pl": "Nie udało mi się znaleźć odpowiedzi na to pytanie na oficjalnych stronach GOV.UK i NHS. "
          "Sprawdź GOV.UK lub NHS.uk albo skontaktuj się z lokalnym urzędem (council).",
    "ro": "Nu am găsit un răspuns la această întrebare în paginile oficiale GOV.UK și NHS. "
          "Vă rugăm să verificați GOV.UK sau NHS.uk ori să contactați consiliul local.",
    "ar": "لم أجد إجابة عن هذا السؤال في صفحات GOV.UK وNHS الرسمية. "
          "يُرجى مراجعة GOV.UK أو NHS.uk، أو التواصل مع المجلس المحلي.",
    "ur": "مجھے سرکاری GOV.UK اور NHS صفحات میں اس سوال کا جواب نہیں ملا۔ "
          "براہ کرم GOV.UK یا NHS.uk دیکھیں، یا اپنی مقامی کونسل سے رابطہ کریں۔",
}
HEALTH_NOTE = {
    "en": "If you need medical help now, call 111. In an emergency, call 999.",
    "pl": "Jeśli potrzebujesz pilnej pomocy medycznej, zadzwoń pod numer 111. W nagłym wypadku dzwoń pod 999.",
    "ro": "Dacă aveți nevoie urgent de ajutor medical, sunați la 111. În caz de urgență, sunați la 999.",
    "ar": "إذا كنت بحاجة إلى مساعدة طبية الآن، اتصل بالرقم 111. في حالات الطوارئ، اتصل بالرقم 999.",
    "ur": "اگر آپ کو ابھی طبی مدد کی ضرورت ہے تو 111 پر کال کریں۔ ہنگامی صورت میں 999 پر کال کریں۔",
}


def translate_answer(translator, english, lang):
    """Translate an English answer sentence by sentence, keeping each sentence's [n] citations.

    Citation markers are taken out before translation (NLLB may drop or garble them) and put back
    at the end of the translated sentence."""
    out = []
    for sent in split_sentences(english):
        marks = "".join(f"[{n}]" for n in CITATION.findall(sent))
        text = CITATION.sub("", sent).replace(" .", ".").strip()
        if text:
            out.append((translator.translate([text], src="en", tgt=lang)[0] + " " + marks).strip())
        elif marks and out:  # markers split off on their own belong to the previous sentence
            out[-1] += " " + marks
    return " ".join(out)


class Assistant:
    def __init__(self, pipeline=None, generator=None, threshold=THRESHOLD, mode="direct"):
        self.pipeline = pipeline or SearchPipeline()
        self.generator = generator or Generator()
        self.threshold = threshold
        self.mode = mode

    def refuse(self, base, reason):
        return {**base, "refused": True, "reason": reason, "answer": REFUSAL[base["lang"]], "health_note": None,
                "sources": []}

    def answer_from(self, search, question, mode=None):
        """Answer from an existing search result (lets the evaluation reuse one search for both modes)."""
        mode = mode or self.mode
        lang, passages = search["lang"], search["results"]
        top = passages[0]["score"] if passages else 0.0
        base = {"question": question, "lang": lang, "mode": mode, "english_query": search["english_query"],
                "top_score": top, "raw": ""}
        if top < self.threshold:
            return self.refuse(base, "low relevance")
        if not self.generator.answerable(question, passages):
            return self.refuse(base, "check: not in sources")
        answer_lang = lang if mode == "direct" else "en"
        raw = self.generator.answer(question, passages, answer_lang, pivot=answer_lang != lang)
        base["raw"] = raw
        if NO_ANSWER in raw:
            return self.refuse(base, "model: not in sources")
        text = raw if answer_lang == lang else translate_answer(self.pipeline.translator, raw, lang)
        cited = [n for n in citations(text) if 1 <= n <= len(passages)]
        sources = [{"n": n, **{k: passages[n - 1][k] for k in ("title", "heading", "url", "source",
                                                                 "public_updated_at", "fetched_at")}}
                   for n in cited]
        # The note stays appended to the answer text (as evaluated in weeks 6-7); the web app shows it apart.
        note = HEALTH_NOTE[lang] if any(s["source"] == "NHS.uk" for s in sources) else None
        if note:
            text += "\n\n" + note
        return {**base, "refused": False, "reason": None, "answer": text, "health_note": note,
                "sources": sources, "passages": passages}

    def ask(self, question, lang=None, mode=None):
        return self.answer_from(self.pipeline.search(question, k=3, lang=lang), question, mode)


In [ ]:
%%writefile src/app.py
"""Week 8 web front-end (task T34): the week 6 assistant behind a small local web page.

    browser (web/index.html, app.js, style.css, i18n.json)
      -> POST /api/ask {"question": "..."}
      -> Assistant.ask (assistant.py, unchanged pipeline: direct mode, threshold 0.0052, yes/no check)
      -> JSON: answer, answer language, refusal flag, health note, source cards with dates

Design choices (see docs/week-08-report.md):
  * Python standard library only (http.server): no new dependencies, nothing to deploy.
  * Binds to 127.0.0.1 by default and calls no outside service, so questions never leave the machine
    (docs/ethics-notes.md). Questions are not logged or stored: the request log has method, path and
    status only.
  * The models take a minute or two to load, so the page is served at once and the assistant loads in
    a background thread; /api/status tells the page when it is ready.
  * One question at a time (a lock): the models are not thread-safe, and a laptop has one GPU.

Run:  .venv/bin/python src/app.py            then open http://127.0.0.1:8000
"""
import argparse
import json
import pathlib
import sys
import threading
import time
import traceback
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

ROOT = pathlib.Path(__file__).resolve().parents[1]
WEB = ROOT / "web"
MAX_CHARS = 500     # longest question accepted; the longest test question is under 200 characters
MAX_BODY = 8_192    # bytes; 500 Arabic or Urdu characters are about 1 KB of UTF-8
TYPES = {".html": "text/html; charset=utf-8", ".css": "text/css; charset=utf-8",
         ".js": "text/javascript; charset=utf-8", ".json": "application/json; charset=utf-8",
         ".svg": "image/svg+xml"}
SECURITY_HEADERS = {
    # Everything is served from this machine: no fonts, scripts or trackers from elsewhere.
    "Content-Security-Policy": "default-src 'self'; img-src 'self' data:; object-src 'none'; "
                               "base-uri 'none'; frame-ancestors 'none'; form-action 'self'",
    "X-Content-Type-Options": "nosniff",
    "Referrer-Policy": "no-referrer",
}
SOURCE_KEYS = ("n", "title", "heading", "url", "source", "public_updated_at", "fetched_at")


class Backend:
    """Loads the assistant in a background thread and answers one question at a time."""

    def __init__(self, factory):
        self.factory = factory
        self.assistant = None
        self.error = None
        self.loaded = threading.Event()
        self.lock = threading.Lock()

    def start(self):
        threading.Thread(target=self._load, name="load-assistant", daemon=True).start()
        return self

    def _load(self):
        t0 = time.time()
        try:
            self.assistant = self.factory()
            print(f"assistant ready in {time.time() - t0:.0f} s", flush=True)
        except Exception as e:  # shown on /api/status so the page can say the assistant is unavailable
            self.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        finally:
            self.loaded.set()

    def status(self):
        if not self.loaded.is_set():
            return "loading"
        return "error" if self.error else "ready"

    def ask(self, question):
        with self.lock:
            t0 = time.time()
            result = self.assistant.ask(question)
            return public(result, time.time() - t0)


def public(result, seconds=None):
    """What the page needs from an Assistant result. Raw model output, passages and the refusal reason
    stay on the server: the page only shows the answer and the official pages behind it."""
    note = result.get("health_note")
    answer = result["answer"]
    if note:
        answer = answer.removesuffix("\n\n" + note)
    sources, cited = result.get("sources", []), True
    if not result["refused"] and not sources:
        # About 4% of answers carry no [n] markers (6/150 in week 6, 10/248 in week 7). Every answer must still link to the official
        # pages it was written from (docs/ethics-notes.md), so show the passages given to the model.
        sources, cited = [{**p, "n": None} for p in result.get("passages", [])], False
    return {"lang": result["lang"], "refused": bool(result["refused"]), "answer": answer, "health_note": note,
            "cited": cited, "sources": [{k: s.get(k) for k in SOURCE_KEYS} for s in sources],
            "seconds": None if seconds is None else round(seconds, 1)}


def validate(body):
    """Return (question, None) or (None, error code). Codes match the page's i18n keys."""
    try:
        data = json.loads(body.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError):
        return None, "bad_request"
    question = data.get("question") if isinstance(data, dict) else None
    if not isinstance(question, str):
        return None, "bad_request"
    question = " ".join(question.split())  # collapse newlines and repeated spaces
    if not question:
        return None, "empty"
    if len(question) > MAX_CHARS:
        return None, "too_long"
    return question, None


def make_handler(backend):
    class Handler(BaseHTTPRequestHandler):
        server_version = "civic-assistant"
        sys_version = ""

        def log_message(self, fmt, *args):
            pass  # replaced by log_request below, which never includes the question

        def log_request(self, code="-", size="-"):
            code = getattr(code, "value", code)
            print(f"{time.strftime('%H:%M:%S')} {self.command} {self.path.split('?')[0]} {code}", flush=True)

        def send(self, status, body, ctype, cache="no-store"):
            self.send_response(status)
            self.send_header("Content-Type", ctype)
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", cache)
            for k, v in SECURITY_HEADERS.items():
                self.send_header(k, v)
            self.end_headers()
            if self.command != "HEAD":
                self.wfile.write(body)

        def send_json(self, status, obj):
            self.send(status, json.dumps(obj, ensure_ascii=False).encode("utf-8"), TYPES[".json"])

        def do_HEAD(self):
            self.do_GET()

        def do_GET(self):
            path = self.path.split("?")[0]
            if path == "/api/status":
                return self.send_json(HTTPStatus.OK, {"status": backend.status(), "max_chars": MAX_CHARS})
            name = "index.html" if path in ("/", "/index.html") else path.removeprefix("/static/")
            if name == path:  # neither the page nor /static/...
                return self.send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})
            file = (WEB / name).resolve()
            if file.parent != WEB.resolve() or file.suffix not in TYPES or not file.is_file():
                return self.send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})
            self.send(HTTPStatus.OK, file.read_bytes(), TYPES[file.suffix], cache="no-cache")

        def do_POST(self):
            if self.path.split("?")[0] != "/api/ask":
                return self.send_json(HTTPStatus.NOT_FOUND, {"error": "not_found"})
            try:
                length = int(self.headers.get("Content-Length", "0"))
            except ValueError:
                length = -1
            if not 0 < length <= MAX_BODY:
                return self.send_json(HTTPStatus.BAD_REQUEST, {"error": "too_long" if length > MAX_BODY
                                                               else "bad_request"})
            question, error = validate(self.rfile.read(length))
            if error:
                return self.send_json(HTTPStatus.BAD_REQUEST, {"error": error})
            status = backend.status()
            if status != "ready":
                return self.send_json(HTTPStatus.SERVICE_UNAVAILABLE, {"error": status})
            try:
                return self.send_json(HTTPStatus.OK, backend.ask(question))
            except Exception:
                traceback.print_exc()  # the traceback has no question text in it
                return self.send_json(HTTPStatus.INTERNAL_SERVER_ERROR, {"error": "server"})

    return Handler


def make_server(backend, host="127.0.0.1", port=8000):
    return ThreadingHTTPServer((host, port), make_handler(backend))


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--host", default="127.0.0.1", help="default 127.0.0.1: only this computer can connect")
    ap.add_argument("--port", type=int, default=8000)
    ap.add_argument("--mode", choices=("direct", "translate"), default="direct",
                    help="answer language: direct (chosen in week 6) or NLLB back-translation")
    args = ap.parse_args(argv)
    if args.host not in ("127.0.0.1", "localhost", "::1"):
        print(f"warning: listening on {args.host}; questions travel over the network unencrypted", file=sys.stderr)

    def load():
        sys.path.insert(0, str(ROOT / "src"))
        import torch  # heavy imports (torch, models) only once the page is up
        from assistant import Assistant
        from pipeline import SearchPipeline
        # NLLB on the Apple GPU when there is one: about 0.3 s per question instead of minutes on a busy CPU.
        device = None if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else None
        return Assistant(pipeline=SearchPipeline(translator_device=device), mode=args.mode)

    backend = Backend(load).start()
    server = make_server(backend, args.host, args.port)
    print(f"serving on http://{args.host}:{args.port}  (models loading in the background)", flush=True)
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass
    finally:
        server.server_close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/build_queries.py
"""Build the retrieval test set and check every answer phrase against the corpus.

Each English question has:
  gold_url - the page that answers it
  evidence - a short phrase copied verbatim from that page that contains the answer
A retrieved chunk only counts as "answer-bearing" if it contains the evidence
phrase, so a hit on the right page but the wrong section is not counted.

Translations (pl, ro, ar, ur) were written by the author and are flagged
"unverified" until a native speaker checks them.
Output: eval/queries.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"

# qid: (domain, gold_url, evidence, {lang: question})
Q = {
    "S01": ("schools", G + "schools-admissions", "close on 15 January", {
        "en": "When is the deadline to apply for a primary school place?",
        "pl": "Jaki jest termin składania wniosku o miejsce w szkole podstawowej?",
        "ro": "Care este termenul limită pentru a depune cererea pentru un loc la școala primară?",
        "ar": "ما هو الموعد النهائي للتقديم على مقعد في المدرسة الابتدائية؟",
        "ur": "پرائمری اسکول میں داخلے کی درخواست دینے کی آخری تاریخ کیا ہے؟"}),
    "S02": ("schools", G + "schools-admissions", "by 31 October", {
        "en": "What is the last date to apply for a secondary school place?",
        "pl": "Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?",
        "ro": "Până la ce dată trebuie să aplic pentru un loc la școala secundară?",
        "ar": "ما هو آخر موعد للتقديم على مقعد في المدرسة الثانوية؟",
        "ur": "سیکنڈری اسکول میں داخلے کے لیے درخواست کس تاریخ تک دینی ہوگی؟"}),
    "S03": ("schools", G + "schools-admissions", "primary schools on 16 April", {
        "en": "When will I find out which primary school my child has been offered?",
        "pl": "Kiedy dowiem się, do której szkoły podstawowej zostało przyjęte moje dziecko?",
        "ro": "Când voi afla la ce școală primară a primit loc copilul meu?",
        "ar": "متى سأعرف المدرسة الابتدائية التي قُبل فيها طفلي؟",
        "ur": "مجھے کب پتہ چلے گا کہ میرے بچے کو کون سے پرائمری اسکول میں جگہ ملی ہے؟"}),
    "S04": ("schools", G + "schools-admissions", "at least 20 school days", {
        "en": "How long do I have to appeal if my child is refused a school place?",
        "pl": "Ile mam czasu na odwołanie, jeśli moje dziecko nie dostało miejsca w szkole?",
        "ro": "Cât timp am la dispoziție să fac contestație dacă copilul meu nu a primit loc la școală?",
        "ar": "كم لدي من الوقت للاستئناف إذا رُفض طلب طفلي للحصول على مقعد في المدرسة؟",
        "ur": "اگر میرے بچے کو اسکول میں جگہ نہ ملے تو اپیل کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    "S05": ("schools", G + "school-attendance-absence", "fined or face legal action for taking your child on holiday", {
        "en": "Can I be fined for taking my child on holiday during term time?",
        "pl": "Czy mogę dostać grzywnę za zabranie dziecka na wakacje w trakcie roku szkolnego?",
        "ro": "Pot primi amendă dacă îmi iau copilul în vacanță în timpul semestrului?",
        "ar": "هل يمكن أن أُغرَّم إذا أخذت طفلي في عطلة أثناء الفصل الدراسي؟",
        "ur": "کیا ٹرم کے دوران بچے کو چھٹیوں پر لے جانے پر مجھے جرمانہ ہو سکتا ہے؟"}),
    "S06": ("schools", G + "apply-free-school-meals", "free school meals if you get any of the following", {
        "en": "Can my child get free school meals if I receive Universal Credit?",
        "pl": "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?",
        "ro": "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?",
        "ar": "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟",
        "ur": "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S07": ("schools", G + "free-school-transport", "more than 2 miles away and the child is under 8", {
        "en": "Is school transport free if the school is far from home?",
        "pl": "Czy transport do szkoły jest darmowy, jeśli szkoła jest daleko od domu?",
        "ro": "Este transportul la școală gratuit dacă școala este departe de casă?",
        "ar": "هل المواصلات المدرسية مجانية إذا كانت المدرسة بعيدة عن المنزل؟",
        "ur": "اگر اسکول گھر سے دور ہو تو کیا اسکول کی ٹرانسپورٹ مفت ہے؟"}),
    "S08": ("schools", G + "know-when-you-can-leave-school", "16 by the end of the summer holidays", {
        "en": "At what age can my child leave school?",
        "pl": "W jakim wieku moje dziecko może skończyć szkołę?",
        "ro": "La ce vârstă poate copilul meu să termine școala?",
        "ar": "في أي عمر يمكن لطفلي ترك المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول چھوڑ سکتا ہے؟"}),
    "S09": ("schools", G + "children-with-special-educational-needs", "SEN support - support given in school", {
        "en": "What support can my child get for special educational needs?",
        "pl": "Jakie wsparcie może otrzymać moje dziecko ze specjalnymi potrzebami edukacyjnymi?",
        "ro": "Ce sprijin poate primi copilul meu pentru cerințe educaționale speciale?",
        "ar": "ما الدعم الذي يمكن أن يحصل عليه طفلي إذا كانت لديه احتياجات تعليمية خاصة؟",
        "ur": "خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟"}),
    "S10": ("schools", G + "complain-about-school", "different ways to complain in England", {
        "en": "How do I complain about my child's state school?",
        "pl": "Jak złożyć skargę na państwową szkołę mojego dziecka?",
        "ro": "Cum fac o plângere împotriva școlii de stat a copilului meu?",
        "ar": "كيف أقدم شكوى ضد المدرسة الحكومية التي يدرس فيها طفلي؟",
        "ur": "میں اپنے بچے کے سرکاری اسکول کے بارے میں شکایت کیسے کروں؟"}),

    "H01": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "section 21 process", {
        "en": "Can my landlord evict me without giving a reason?",
        "pl": "Czy wynajmujący może mnie eksmitować bez podania powodu?",
        "ro": "Poate proprietarul să mă evacueze fără să-mi dea un motiv?",
        "ar": "هل يمكن للمالك أن يطردني دون إبداء سبب؟",
        "ur": "کیا مالک مکان مجھے بغیر کوئی وجہ بتائے گھر سے نکال سکتا ہے؟"}),
    "H02": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "increase the rent once a year", {
        "en": "How often can my landlord increase the rent?",
        "pl": "Jak często wynajmujący może podnosić czynsz?",
        "ro": "Cât de des poate proprietarul să crească chiria?",
        "ar": "كم مرة يمكن للمالك أن يرفع الإيجار؟",
        "ur": "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟"}),
    "H03": ("housing", G + "tenancy-deposit-protection", "Deposit Protection Service", {
        "en": "My landlord took a deposit from me. Does it have to be protected?",
        "pl": "Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?",
        "ro": "Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?",
        "ar": "أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟",
        "ur": "مالک مکان نے مجھ سے ڈپازٹ لیا ہے، کیا اس ڈپازٹ کا محفوظ ہونا ضروری ہے؟"}),
    "H04": ("housing", G + "housing-benefit", "help you pay your rent", {
        "en": "How can I get help paying my rent if I am on a low income?",
        "pl": "Jak uzyskać pomoc w opłaceniu czynszu, jeśli mam niskie dochody?",
        "ro": "Cum pot primi ajutor pentru plata chiriei dacă am un venit mic?",
        "ar": "كيف أحصل على مساعدة في دفع الإيجار إذا كان دخلي منخفضًا؟",
        "ur": "کم آمدنی کی صورت میں مجھے کرایہ ادا کرنے میں مدد کیسے مل سکتی ہے؟"}),
    "H05": ("housing", G + "council-housing", "join a waiting list", {
        "en": "How do I apply for council housing?",
        "pl": "Jak złożyć wniosek o mieszkanie komunalne?",
        "ro": "Cum aplic pentru o locuință socială de la consiliul local?",
        "ar": "كيف أتقدم بطلب للحصول على سكن من المجلس المحلي؟",
        "ur": "میں کونسل کے مکان کے لیے درخواست کیسے دوں؟"}),
    "H06": ("housing", G + "homelessness-help-from-council", "homeless or about to lose your home", {
        "en": "I am about to lose my home. Can the council help me?",
        "pl": "Zaraz stracę dom. Czy lokalny urząd może mi pomóc?",
        "ro": "Sunt pe punctul de a-mi pierde locuința. Mă poate ajuta consiliul local?",
        "ar": "أنا على وشك أن أفقد منزلي، هل يمكن للمجلس المحلي مساعدتي؟",
        "ur": "میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟"}),
    "H07": ("housing", G + "private-renting", "safe and in a good state of repair", {
        "en": "Does my landlord have to keep my rented home in good repair?",
        "pl": "Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?",
        "ro": "Este proprietarul obligat să mențină locuința închiriată în stare bună?",
        "ar": "هل يجب على المالك أن يحافظ على المنزل المؤجر في حالة جيدة؟",
        "ur": "کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟"}),
    "H08": ("housing", G + "private-renting", "24 hours", {
        "en": "How much notice must my landlord give before visiting the property?",
        "pl": "Z jakim wyprzedzeniem wynajmujący musi mnie uprzedzić przed wizytą w mieszkaniu?",
        "ro": "Cu cât timp înainte trebuie să mă anunțe proprietarul dacă vrea să vină în locuință?",
        "ar": "كم من الوقت يجب أن يخطرني المالك قبل أن يزور المنزل؟",
        "ur": "گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟"}),
    "H09": ("housing", G + "council-tax", "25% off your bill", {
        "en": "Do I get a council tax discount if I live alone?",
        "pl": "Czy dostanę zniżkę na podatek lokalny, jeśli mieszkam sam?",
        "ro": "Primesc reducere la impozitul local pe locuință dacă locuiesc singur?",
        "ar": "هل أحصل على خصم في ضريبة المجلس المحلي إذا كنت أعيش وحدي؟",
        "ur": "اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟"}),
    "H10": ("housing", G + "council-housing-association-evictions", "stop or delay the eviction", {
        "en": "What can I do if my council or housing association wants to evict me?",
        "pl": "Co mogę zrobić, jeśli lokalny urząd lub stowarzyszenie mieszkaniowe chce mnie eksmitować?",
        "ro": "Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?",
        "ar": "ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟",
        "ur": "اگر کونسل یا ہاؤسنگ ایسوسی ایشن مجھے گھر سے نکالنا چاہے تو میں کیا کر سکتا ہوں؟"}),

    "M01": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "change your current GP surgery online", {
        "en": "How do I register with a GP surgery?",
        "pl": "Jak zarejestrować się u lekarza rodzinnego?",
        "ro": "Cum mă înregistrez la un medic de familie?",
        "ar": "كيف أسجل لدى طبيب عام؟",
        "ur": "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟"}),
    "M02": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "do not need ID, proof of address", {
        "en": "Do I need proof of address or immigration status to register with a GP?",
        "pl": "Czy potrzebuję potwierdzenia adresu lub statusu imigracyjnego, aby zarejestrować się u lekarza rodzinnego?",
        "ro": "Am nevoie de dovada adresei sau a statutului de imigrant ca să mă înregistrez la medicul de familie?",
        "ar": "هل أحتاج إلى إثبات عنوان أو إثبات وضع الهجرة للتسجيل لدى طبيب عام؟",
        "ur": "کیا جی پی کے پاس رجسٹر ہونے کے لیے مجھے پتے یا امیگریشن اسٹیٹس کا ثبوت چاہیے؟"}),
    "M03": ("health", N + "urgent-and-emergency-care-services/when-to-use-111/", "if you think you need medical help right now", {
        "en": "When should I use NHS 111?",
        "pl": "Kiedy powinienem skorzystać z NHS 111?",
        "ro": "Când ar trebui să folosesc NHS 111?",
        "ar": "متى يجب أن أستخدم خدمة NHS 111؟",
        "ur": "مجھے NHS 111 کب استعمال کرنا چاہیے؟"}),
    "M04": ("health", N + "urgent-and-emergency-care-services/when-to-go-to-ae/", "serious injuries and life-threatening emergencies", {
        "en": "When should I go to A&E?",
        "pl": "Kiedy powinienem iść na szpitalny oddział ratunkowy?",
        "ro": "Când ar trebui să merg la urgențe?",
        "ar": "متى يجب أن أذهب إلى قسم الطوارئ؟",
        "ur": "مجھے ہسپتال کے ایمرجنسی شعبے میں کب جانا چاہیے؟"}),
    "M05": ("health", N + "urgent-and-emergency-care-services/when-to-call-999/", "999 is for life-threatening emergencies", {
        "en": "When should I call 999?",
        "pl": "Kiedy powinienem dzwonić pod numer 999?",
        "ro": "Când ar trebui să sun la 999?",
        "ar": "متى يجب أن أتصل بالرقم 999؟",
        "ur": "مجھے 999 پر کب کال کرنی چاہیے؟"}),
    "M06": ("health", N + "dentists/how-to-find-an-nhs-dentist/", "Ask for an NHS appointment", {
        "en": "How can I find an NHS dentist?",
        "pl": "Jak znaleźć dentystę NHS?",
        "ro": "Cum găsesc un dentist NHS?",
        "ar": "كيف أجد طبيب أسنان تابعًا لـ NHS؟",
        "ur": "میں NHS کا ڈینٹسٹ کیسے تلاش کروں؟"}),
    "M07": ("health", N + "dentists/how-much-nhs-dental-treatment-costs/", "£27.90", {
        "en": "How much does a basic NHS dental check-up cost?",
        "pl": "Ile kosztuje podstawowy przegląd u dentysty w NHS?",
        "ro": "Cât costă un control stomatologic de bază la NHS?",
        "ar": "كم يكلف فحص الأسنان الأساسي في NHS؟",
        "ur": "NHS میں دانتوں کے بنیادی معائنے پر کتنا خرچ آتا ہے؟"}),
    "M08": ("health", N + "pharmacies/how-pharmacies-can-help/", "This is called Pharmacy First", {
        "en": "Can a pharmacy give me prescription medicine without seeing a GP?",
        "pl": "Czy apteka może wydać mi lek na receptę bez wizyty u lekarza?",
        "ro": "Poate o farmacie să-mi dea medicamente cu rețetă fără să merg la medic?",
        "ar": "هل يمكن للصيدلية أن تعطيني دواءً يحتاج إلى وصفة طبية دون أن أرى الطبيب؟",
        "ur": "کیا فارمیسی ڈاکٹر کو دکھائے بغیر مجھے نسخے والی دوا دے سکتی ہے؟"}),
    "M09": ("health", N + "help-with-health-costs/nhs-low-income-scheme-lis/", "HC2 certificate", {
        "en": "Can I get help with health costs if I have a low income?",
        "pl": "Czy mogę dostać pomoc w pokryciu kosztów leczenia, jeśli mam niskie dochody?",
        "ro": "Pot primi ajutor pentru costurile medicale dacă am venituri mici?",
        "ar": "هل يمكنني الحصول على مساعدة في التكاليف الصحية إذا كان دخلي منخفضًا؟",
        "ur": "کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟"}),
    "M10": ("health", G + "healthcare-immigration-application", "£776 per year for students", {
        "en": "How much is the immigration health surcharge for students?",
        "pl": "Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?",
        "ro": "Cât costă taxa de sănătate pentru imigranți în cazul studenților?",
        "ar": "كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟",
        "ur": "طلبہ کے لیے امیگریشن ہیلتھ سرچارج کتنا ہے؟"}),
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url) for q, (_, url, ev, _) in Q.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit(f"evidence not found on gold page: {bad}")
    out = ROOT / "eval/queries.csv"
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, qs) in Q.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    print(f"{len(Q)} questions x {len(next(iter(Q.values()))[3])} languages -> {out.relative_to(ROOT)}; "
          f"all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile web/index.html
<!doctype html>
<html lang="en" dir="ltr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Ask about health, housing and schools</title>
<link rel="icon" href="data:,">
<link rel="stylesheet" href="/static/style.css">
<script src="/static/app.js" defer></script>
</head>
<body>
<a class="skip-link" href="#question" data-i18n="skip">Skip to your question</a>

<header class="masthead">
  <div class="wrap masthead-inner">
    <p class="product"><span lang="en" dir="ltr">GOV.UK</span> · <span lang="en" dir="ltr">NHS.uk</span></p>
    <div class="lang-picker">
      <label for="lang" data-i18n="lang_label">Language</label>
      <select id="lang" autocomplete="off"></select>
    </div>
  </div>
</header>

<main id="main" class="wrap">
  <h1 data-i18n="title">Ask about health, housing and schools in England</h1>
  <p class="intro" data-i18n="intro"></p>

  <div class="notice" role="note">
    <p><strong data-i18n="notice_title"></strong> <span data-i18n="notice"></span></p>
  </div>

  <form id="ask-form" class="ask" novalidate>
    <label for="question" class="label" data-i18n="q_label">Your question</label>
    <p id="question-hint" class="hint" data-i18n="q_hint"></p>
    <p id="question-error" class="error" hidden></p>
    <textarea id="question" name="question" rows="3" maxlength="500" dir="auto" spellcheck="true"
              autocomplete="off" aria-describedby="question-hint"></textarea>
    <button type="submit" id="ask" class="button" data-i18n="ask">Ask</button>
  </form>

  <p id="status" class="status" role="status" aria-live="polite"></p>

  <section id="result" class="result" aria-labelledby="result-heading" hidden>
    <h2 id="result-heading" tabindex="-1"></h2>
    <p id="answer-lang" class="meta"></p>
    <div id="answer" class="answer"></div>
    <div id="health" class="health" role="note" hidden>
      <h3 data-i18n="health_heading"></h3>
      <p id="health-text"></p>
    </div>
    <div id="official" class="official" hidden>
      <h3 data-i18n="official_sites"></h3>
      <ul>
        <li><a href="https://www.gov.uk" target="_blank" rel="noopener noreferrer"><bdi lang="en" dir="ltr">www.gov.uk</bdi><span class="visually-hidden" data-i18n="new_tab"></span></a></li>
        <li><a href="https://www.nhs.uk" target="_blank" rel="noopener noreferrer"><bdi lang="en" dir="ltr">www.nhs.uk</bdi><span class="visually-hidden" data-i18n="new_tab"></span></a></li>
      </ul>
    </div>
    <div id="sources-block" hidden>
      <h3 id="sources-heading" data-i18n="sources_heading"></h3>
      <p id="sources-intro" class="hint"></p>
      <ol id="sources" class="sources" aria-labelledby="sources-heading"></ol>
    </div>
  </section>

  <section class="examples" aria-labelledby="examples-heading">
    <h2 id="examples-heading" data-i18n="examples">Example questions</h2>
    <p class="hint" data-i18n="examples_hint"></p>
    <ul id="examples"></ul>
  </section>
</main>

<footer class="footer">
  <div class="wrap">
    <p class="help" data-i18n="help"></p>
    <p class="licence" lang="en" dir="ltr" data-i18n="licence"></p>
  </div>
</footer>

<noscript><p class="wrap">This page needs JavaScript.</p></noscript>
</body>
</html>


In [ ]:
%%writefile web/app.js
// Week 8 front-end (T34): language picker, right-to-left layout, cited answers and source cards.
// Model output is only ever inserted with textContent, never innerHTML.
"use strict";
(() => {
  const $ = (id) => document.getElementById(id);
  const BULLET = /^\s*(?:[-*•]|\d+[.)])\s+/;
  const CITE = /\[(\d+)\]/g;

  let T = null;            // i18n.json
  let ui = "en";           // interface language
  let maxChars = 500;
  let backend = "loading"; // loading | ready | error | unreachable
  let busy = false;
  let statusKey = null;    // kept so messages can be re-translated when the language changes
  let errorKey = null;
  let last = null;         // last answer, redrawn when the language changes

  const fill = (s, vars) => s.replace(/\{(\w+)\}/g, (m, k) => (k in vars ? String(vars[k]) : m));
  const t = (key, vars = {}) => fill((T && (T[ui]?.[key] ?? T.en[key])) ?? key, { max: maxChars, ...vars });
  const dirOf = (lang) => (T.rtl.includes(lang) ? "rtl" : "ltr");
  const el = (tag, props = {}, ...children) => {
    const node = Object.assign(document.createElement(tag), props);
    node.append(...children);
    return node;
  };

  // ---------- language ----------
  function remembered() { try { return localStorage.getItem("ui-lang"); } catch { return null; } }
  function remember(lang) { try { localStorage.setItem("ui-lang", lang); } catch { /* private mode */ } }

  function initialLanguage() {
    const fromUrl = new URLSearchParams(location.search).get("lang");  // e.g. study links: /?lang=ur
    for (const cand of [fromUrl, remembered(), ...(navigator.languages || [])]) {
      const base = (cand || "").toLowerCase().split("-")[0];
      if (T.order.includes(base)) return base;
    }
    return "en";
  }

  function applyLanguage(lang) {
    ui = lang;
    const root = document.documentElement;
    root.lang = lang;
    root.dir = dirOf(lang);
    for (const node of document.querySelectorAll("[data-i18n]")) node.textContent = t(node.dataset.i18n);
    $("lang").value = lang;
    $("question").maxLength = maxChars;
    renderExamples();
    if (statusKey) setStatus(statusKey, $("status").classList.contains("busy"));
    if (errorKey) showError(errorKey, false);
    if (last) render(last, false);
    else document.title = t("page_title");
  }

  function buildPicker() {
    const select = $("lang");
    select.replaceChildren(...T.order.map((code) =>
      el("option", { value: code, lang: code, dir: dirOf(code), textContent: T[code].name })));
    select.addEventListener("change", () => { remember(select.value); applyLanguage(select.value); });
  }

  // ---------- status and errors ----------
  function setStatus(key, spinning = false) {
    statusKey = key;
    const node = $("status");
    node.textContent = key ? t(key) : "";
    node.classList.toggle("busy", spinning);
  }

  function showError(code, focus = true) {
    errorKey = code;
    const box = $("question-error");
    box.textContent = t(`err_${code}`);
    box.hidden = false;
    const q = $("question");
    q.setAttribute("aria-invalid", "true");
    q.setAttribute("aria-describedby", "question-error question-hint");
    if (focus) q.focus();
  }

  function clearError() {
    errorKey = null;
    $("question-error").hidden = true;
    const q = $("question");
    q.removeAttribute("aria-invalid");
    q.setAttribute("aria-describedby", "question-hint");
  }

  async function fetchStatus() {
    try {
      const r = await fetch("/api/status", { cache: "no-store" });
      const j = await r.json();
      backend = j.status;
      if (j.max_chars && j.max_chars !== maxChars) { maxChars = j.max_chars; applyLanguage(ui); }
    } catch {
      backend = "unreachable";
    }
  }

  async function poll() {
    await fetchStatus();
    if (busy) return;
    if (backend === "loading") { setStatus("starting", true); setTimeout(poll, 3000); }
    else if (backend === "ready") { if (statusKey === "starting" || statusKey === "err_network") setStatus("ready"); }
    else if (backend === "error") setStatus("err_error");
    else { setStatus("err_network"); setTimeout(poll, 5000); }
  }

  // ---------- asking ----------
  async function ask(event) {
    event.preventDefault();
    if (busy) return;
    const question = $("question").value.trim();
    if (!question) return showError("empty");
    if (question.length > maxChars) return showError("too_long");
    clearError();
    if (backend !== "ready") await fetchStatus();  // the first status check may not have returned yet
    if (backend === "loading") return setStatus("err_loading");

    busy = true;
    $("ask").setAttribute("aria-disabled", "true");
    $("result").setAttribute("aria-busy", "true");
    setStatus("searching", true);
    const abort = new AbortController();
    const timer = setTimeout(() => abort.abort(), 5 * 60 * 1000);
    try {
      const r = await fetch("/api/ask", {
        method: "POST", headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ question }), signal: abort.signal,
      });
      const j = await r.json().catch(() => null);
      if (!r.ok) {
        const code = j?.error || "server";
        if (code === "empty" || code === "too_long") { setStatus(null); showError(code); }
        else setStatus(`err_${code}` in T.en ? `err_${code}` : "err_server");
        if (code === "loading") { backend = "loading"; setTimeout(poll, 3000); }
        return;
      }
      last = j;
      setStatus(null);
      render(j, true);
    } catch {
      setStatus("err_network");
    } finally {
      clearTimeout(timer);
      busy = false;
      $("ask").removeAttribute("aria-disabled");
      $("result").removeAttribute("aria-busy");
    }
  }

  // ---------- rendering ----------
  function formatDate(value) {
    if (!value) return null;
    const d = new Date(/^\d{4}-\d{2}-\d{2}$/.test(value) ? `${value}T12:00:00Z` : value);
    if (Number.isNaN(d.getTime())) return value;
    try {
      const locale = ui === "en" ? "en-GB" : ui;  // "28 October 2026", not the US order
      return new Intl.DateTimeFormat(locale, { dateStyle: "long", timeZone: "Europe/London", numberingSystem: "latn" })
        .format(d);
    } catch {
      return d.toISOString().slice(0, 10);
    }
  }

  function withCitations(node, text, valid) {
    let at = 0;
    for (const m of text.matchAll(CITE)) {
      node.append(text.slice(at, m.index));
      const n = Number(m[1]);
      if (valid.has(n)) {
        const link = el("a", { href: `#source-${n}`, className: "cite", textContent: `[${n}]` });
        link.setAttribute("aria-label", t("source_n", { n }));
        node.append(el("bdi", { dir: "ltr" }, link));
      } else {
        node.append(m[0]);
      }
      at = m.index + m[0].length;
    }
    node.append(text.slice(at));
  }

  function renderAnswer(box, text, valid) {
    box.replaceChildren();
    let list = null;
    let ordered = false;
    for (const raw of text.split("\n")) {
      const line = raw.replace(/\*\*/g, "").trim();  // drop markdown bold the model sometimes adds
      if (!line) { list = null; continue; }
      const bullet = line.match(BULLET);
      if (bullet) {
        const isOrdered = /\d/.test(bullet[0]);
        if (!list || ordered !== isOrdered) {
          list = el(isOrdered ? "ol" : "ul");
          ordered = isOrdered;
          box.append(list);
        }
        const item = el("li");
        withCitations(item, line.slice(bullet[0].length), valid);
        list.append(item);
      } else {
        list = null;
        const p = el("p");
        withCitations(p, line, valid);
        box.append(p);
      }
    }
  }

  function sourceCard(s) {
    const card = el("li", { className: "source" });
    const label = el("p", { className: "source-label" });
    if (s.n != null) {  // cited source: target of the [n] links in the answer
      card.id = `source-${s.n}`;
      card.tabIndex = -1;
      label.append(`${t("source_n", { n: s.n })} · `);
    }
    label.append(el("bdi", { lang: "en", dir: "ltr", textContent: s.source }));
    card.append(label);
    card.append(el("h4", { lang: "en", dir: "ltr", textContent: s.title }));
    if (s.heading && s.heading !== s.title) {
      card.append(el("p", { className: "section", lang: "en", dir: "ltr", textContent: s.heading }));
    }
    const dates = el("dl");
    for (const [key, value] of [["updated", s.public_updated_at], ["copied", s.fetched_at]]) {
      const shown = formatDate(value);
      if (shown) dates.append(el("dt", { textContent: t(key) }), el("dd", { textContent: shown }));
    }
    card.append(dates);
    if (/^https:\/\//.test(s.url || "")) {
      const link = el("a", { href: s.url, target: "_blank", rel: "noopener noreferrer", textContent: t("open") });
      link.append(el("span", { className: "visually-hidden", textContent: ` ${t("new_tab")}` }));
      card.append(link, el("span", { className: "url", lang: "en", dir: "ltr",
                                     textContent: s.url.replace(/^https:\/\//, "") }));
    }
    return card;
  }

  function render(res, moveFocus) {
    const valid = new Set(res.sources.map((s) => s.n));
    const heading = t(res.refused ? "refused_heading" : "answer_heading");
    $("result").hidden = false;
    $("result-heading").textContent = heading;
    $("answer-lang").textContent = t("answer_lang", { lang: T[res.lang]?.name || res.lang });

    const answer = $("answer");
    answer.lang = res.lang;
    answer.dir = dirOf(res.lang);
    renderAnswer(answer, res.answer, valid);

    $("health").hidden = !res.health_note;
    const note = $("health-text");
    note.textContent = res.health_note || "";
    note.lang = res.lang;
    note.dir = dirOf(res.lang);

    $("official").hidden = !res.refused;
    $("sources-block").hidden = res.sources.length === 0;
    $("sources-intro").textContent = t(res.cited === false ? "searched_intro" : "sources_intro");
    const byNumber = [...res.sources].sort((a, b) => (a.n ?? 0) - (b.n ?? 0));  // Source 1, 2, 3 in order
    $("sources").replaceChildren(...byNumber.map(sourceCard));

    document.title = `${heading} – ${t("page_title")}`;
    if (moveFocus) $("result-heading").focus();
  }

  function renderExamples() {
    $("examples").replaceChildren(...T[ui].examples_list.map((q) => {
      const b = el("button", { type: "button", className: "example", textContent: q });
      b.addEventListener("click", () => {
        $("question").value = q;
        clearError();
        $("question").focus();
      });
      return el("li", {}, b);
    }));
  }

  // ---------- start ----------
  async function start() {
    try {
      T = await (await fetch("/static/i18n.json")).json();
    } catch {
      $("status").textContent = "Could not load the page text. Reload the page.";
      return;
    }
    buildPicker();
    applyLanguage(initialLanguage());

    $("ask-form").addEventListener("submit", ask);
    $("question").addEventListener("keydown", (e) => {
      if (e.key === "Enter" && (e.ctrlKey || e.metaKey)) $("ask-form").requestSubmit();
    });
    $("question").addEventListener("input", () => { if (errorKey) clearError(); });
    // Citation links move keyboard focus to the source card, not just the scroll position.
    $("answer").addEventListener("click", (e) => {
      const link = e.target.closest("a.cite");
      if (!link) return;
      const card = document.querySelector(link.getAttribute("href"));
      if (!card) return;
      e.preventDefault();
      card.focus();
    });
    poll();
  }

  start();
})();


In [ ]:
%%writefile web/style.css
/* Week 8 front-end styles (T34, T35).
   - Logical properties (margin-inline-start, border-inline-start ...) so the same rules mirror for
     right-to-left Arabic and Urdu: <html dir="rtl"> is all that changes.
   - No web fonts are downloaded (nothing leaves the machine); system fonts per script, with Nastaliq
     for Urdu where the computer has it and Naskh as fallback.
   - Colour pairs are checked for WCAG 2.2 AA contrast in tests/test_web.py: text 4.5:1, borders and
     focus indicators 3:1. Change a colour here and the test tells you if a pair drops below. */
:root {
  color-scheme: light;
  --bg: #ffffff;
  --ink: #0b0c0c;
  --muted: #4c565c;
  --surface: #f3f4f6;
  --line: #6b7280;
  --brand: #26358a;
  --brand-ink: #ffffff;
  --link: #26358a;
  --link-visited: #5b2a86;
  --notice-bg: #fff4d6;
  --notice-line: #8a5a00;
  --health-bg: #e8f0fb;
  --error: #b3261e;
  --focus-ring: #0b0c0c;
  --focus-halo: #ffd84d;
  --font: system-ui, -apple-system, "Segoe UI", Roboto, "Noto Sans", "Helvetica Neue", Arial, sans-serif;
  --font-ar: "Noto Naskh Arabic", "Geeza Pro", "Segoe UI", Tahoma, Arial, sans-serif;
  --font-ur: "Noto Nastaliq Urdu", "Jameel Noori Nastaleeq", "Urdu Typesetting", "Noto Naskh Arabic", "Geeza Pro", serif;
}

*, *::before, *::after { box-sizing: border-box; }
html { -webkit-text-size-adjust: 100%; text-size-adjust: 100%; }
body {
  margin: 0;
  background: var(--bg);
  color: var(--ink);
  font-family: var(--font);
  font-size: 1.125rem;   /* 18px: easier for readers working in a second language */
  line-height: 1.6;
}
:lang(ar) { font-family: var(--font-ar); }
:lang(ur) { font-family: var(--font-ur); line-height: 2.1; }   /* Nastaliq needs tall lines */
:lang(en) { font-family: var(--font); }
:not(html)[lang="en"] { line-height: 1.5; }   /* English titles and links inside an Urdu or Arabic page */

h1, h2, h3 { line-height: 1.3; margin-block: 0 0.75rem; }
:lang(ur) h1, :lang(ur) h2, :lang(ur) h3 { line-height: 1.9; }
h1 { font-size: 1.875rem; }
h2 { font-size: 1.375rem; margin-block-start: 2rem; }
h3 { font-size: 1.125rem; margin-block-start: 1.25rem; }
p { margin-block: 0 1rem; }

.wrap { max-width: 44rem; margin-inline: auto; padding-inline: 1rem; }

a { color: var(--link); text-decoration-thickness: 1px; text-underline-offset: 0.2em; }
a:visited { color: var(--link-visited); }
a:hover { text-decoration-thickness: 3px; }

/* Focus: black ring plus yellow halo, visible on white, grey and the dark blue button and header. */
:focus-visible {
  outline: 3px solid var(--focus-ring);
  outline-offset: 2px;
  box-shadow: 0 0 0 6px var(--focus-halo);
}
[tabindex="-1"]:focus { outline: none; box-shadow: none; }   /* headings that receive focus by script */

.visually-hidden {
  position: absolute !important; width: 1px; height: 1px; padding: 0; margin: -1px;
  overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; border: 0;
}

.skip-link {
  position: absolute; inset-inline-start: 1rem; top: -10rem;
  background: var(--focus-halo); color: var(--ink); padding: 0.5rem 1rem; z-index: 10;
}
.skip-link:focus { top: 0.5rem; }

/* Header with the language picker */
.masthead { background: var(--brand); color: var(--brand-ink); }
.masthead-inner {
  display: flex; flex-wrap: wrap; gap: 0.5rem 1rem; align-items: center; justify-content: space-between;
  padding-block: 0.75rem;
}
.product { margin: 0; font-weight: 600; }
.lang-picker { display: flex; align-items: center; gap: 0.5rem; }
.lang-picker label { font-weight: 600; }
select {
  font: inherit; font-size: 1rem; min-height: 2.75rem; min-width: 9rem;
  padding-block: 0.25rem; padding-inline: 0.5rem;
  color: var(--ink); background: var(--bg); border: 2px solid var(--ink); border-radius: 0;
}

main { padding-block: 1.5rem 2.5rem; }
.intro { font-size: 1.1875rem; }

.notice {
  background: var(--notice-bg);
  border-inline-start: 6px solid var(--notice-line);
  padding: 0.75rem 1rem;
  margin-block-end: 1.5rem;
}
.notice p { margin: 0; }

/* Question form */
.ask { margin-block-end: 1rem; }
.label { display: block; font-weight: 700; font-size: 1.1875rem; margin-block-end: 0.25rem; }
.hint { color: var(--muted); margin-block-end: 0.5rem; }
.error { color: var(--error); font-weight: 700; margin-block-end: 0.5rem; }
textarea {
  display: block; width: 100%; min-height: 6rem; resize: vertical;
  font: inherit; color: var(--ink); background: var(--bg);
  padding: 0.5rem; border: 2px solid var(--ink); border-radius: 0;
  margin-block-end: 1rem;
}
textarea[aria-invalid="true"] { border-color: var(--error); border-width: 3px; }

.button {
  font: inherit; font-weight: 700; min-height: 2.75rem; min-width: 6rem;
  padding-block: 0.5rem; padding-inline: 1.5rem;
  color: var(--brand-ink); background: var(--brand);
  border: 2px solid transparent; border-radius: 0.25rem; cursor: pointer;
}
.button:hover { text-decoration: underline; }
.button[aria-disabled="true"] { cursor: progress; opacity: 0.85; }

.status { min-height: 1.6em; color: var(--muted); }
.status.busy::before {
  content: ""; display: inline-block; width: 0.9em; height: 0.9em; vertical-align: -0.1em;
  margin-inline-end: 0.5rem; border-radius: 50%;
  border: 3px solid var(--line); border-block-start-color: var(--brand);
  animation: spin 0.9s linear infinite;
}
@keyframes spin { to { transform: rotate(360deg); } }

/* Answer */
.result { border-block-start: 4px solid var(--brand); padding-block-start: 1rem; margin-block-end: 2rem; }
.result h2 { margin-block-start: 0; }
.meta { color: var(--muted); font-size: 1rem; }
.answer p { margin-block-end: 0.75rem; }
.answer ul, .answer ol { padding-inline-start: 1.5rem; }
.cite { white-space: nowrap; font-weight: 600; }

.health {
  background: var(--health-bg); border-inline-start: 6px solid var(--brand);
  padding: 0.75rem 1rem; margin-block: 1rem;
}
.health h3 { margin-block-start: 0; }
.health p { margin: 0; font-weight: 600; }

.official ul { padding-inline-start: 1.5rem; }

/* Source cards */
.sources { list-style: none; padding: 0; margin: 0; }
.source {
  background: var(--surface); border: 1px solid var(--line); border-radius: 0.25rem;
  padding: 0.75rem 1rem; margin-block-end: 0.75rem;
}
.source:target, .source:focus { outline: 3px solid var(--focus-ring); box-shadow: 0 0 0 6px var(--focus-halo); }
.source-label { color: var(--muted); font-size: 1rem; font-weight: 700; margin-block-end: 0.25rem; }
.source h4 { font-size: 1.125rem; margin: 0 0 0.25rem; }
.source .section { margin-block-end: 0.5rem; }
.source dl { display: grid; grid-template-columns: max-content 1fr; gap: 0.1rem 0.75rem; margin: 0 0 0.5rem; font-size: 1rem; }
.source dt { color: var(--muted); }
.source dd { margin: 0; }
.source .url { display: block; font-size: 0.9375rem; color: var(--muted); overflow-wrap: anywhere; }

/* Example questions */
.examples ul { list-style: none; padding: 0; margin: 0; }
.examples li { margin-block-end: 0.5rem; }
.example {
  font: inherit; text-align: start; width: 100%; min-height: 2.75rem;
  padding: 0.5rem 0.75rem; color: var(--link); background: var(--bg);
  border: 2px solid var(--line); border-radius: 0.25rem; cursor: pointer;
}
.example:hover { border-color: var(--link); text-decoration: underline; }

.footer { background: var(--surface); border-block-start: 1px solid var(--line); padding-block: 1.25rem; }
.footer p { margin-block-end: 0.5rem; }
.help { font-weight: 700; }
.licence { font-size: 1rem; color: var(--muted); }

@media (max-width: 30rem) {
  body { font-size: 1.0625rem; }
  h1 { font-size: 1.5rem; }
  .masthead-inner { flex-direction: column; align-items: stretch; }
  .lang-picker select { flex: 1; }
  .button { width: 100%; }
  .source dl { grid-template-columns: 1fr; }
  .source dd { margin-block-end: 0.25rem; }
}

@media (prefers-reduced-motion: reduce) {
  .status.busy::before { animation: none; }
}

@media (forced-colors: active) {
  .button, .example, .source { border: 2px solid ButtonText; }
  :focus-visible { outline: 3px solid Highlight; }
}


In [ ]:
%%writefile web/i18n.json
{
  "_note": "Interface text, written by the author; part of the native-speaker check (task T22). 'help' must stay identical to HEALTH_NOTE in src/assistant.py. 'examples' are questions M01, H02 and S06 from eval/queries.csv.",
  "order": [
    "en",
    "pl",
    "ro",
    "ar",
    "ur"
  ],
  "rtl": [
    "ar",
    "ur"
  ],
  "en": {
    "name": "English",
    "page_title": "Ask about health, housing and schools",
    "skip": "Skip to your question",
    "lang_label": "Language",
    "title": "Ask about health, housing and schools in England",
    "intro": "Ask a question in English, Polish, Romanian, Arabic or Urdu. Answers come only from official GOV.UK and NHS.uk pages, with a link to each page.",
    "notice_title": "Research prototype.",
    "notice": "This is a research tool, not official advice. Answers can be wrong or out of date, so always check the linked page. Do not type your name or other personal details.",
    "q_label": "Your question",
    "q_hint": "Write in any of the five languages, up to {max} characters. The answer will be in the language of your question.",
    "ask": "Ask",
    "examples": "Example questions",
    "examples_hint": "Choose one to put it in the question box.",
    "starting": "The assistant is starting. This can take a minute or two.",
    "ready": "The assistant is ready.",
    "searching": "Searching official pages and writing an answer. This can take up to a minute.",
    "answer_heading": "Answer",
    "refused_heading": "No answer found",
    "answer_lang": "Answer language: {lang}",
    "health_heading": "Health help",
    "sources_heading": "Sources",
    "sources_intro": "The answer uses copies of these pages taken on the date shown. Open the page to check for changes.",
    "searched_intro": "This answer does not show which page each sentence comes from. It was written from these pages, so check them.",
    "source_n": "Source {n}",
    "updated": "Page last updated",
    "copied": "Copy taken on",
    "open": "Open the official page",
    "new_tab": "(opens in a new tab)",
    "official_sites": "Official websites",
    "help": "If you need medical help now, call 111. In an emergency, call 999.",
    "licence": "Contains public sector information licensed under the Open Government Licence v3.0.",
    "err_empty": "Type a question.",
    "err_too_long": "Your question is too long. Use up to {max} characters.",
    "err_loading": "The assistant is still starting. Try again in a minute.",
    "err_error": "The assistant could not start. Tell the researcher.",
    "err_server": "Something went wrong. Try again.",
    "err_network": "Could not reach the assistant. Check that it is running, then try again.",
    "err_bad_request": "Something went wrong. Try again.",
    "examples_list": [
      "How do I register with a GP surgery?",
      "How often can my landlord increase the rent?",
      "Can my child get free school meals if I receive Universal Credit?"
    ]
  },
  "pl": {
    "name": "Polski",
    "page_title": "Pytania o zdrowie, mieszkanie i szkołę",
    "skip": "Przejdź do pytania",
    "lang_label": "Język",
    "title": "Pytania o zdrowie, mieszkanie i szkołę w Anglii",
    "intro": "Zadaj pytanie po angielsku, polsku, rumuńsku, arabsku lub w urdu. Odpowiedzi pochodzą wyłącznie z oficjalnych stron GOV.UK i NHS.uk i zawierają link do każdej strony.",
    "notice_title": "Prototyp badawczy.",
    "notice": "To narzędzie badawcze, a nie oficjalna porada. Odpowiedzi mogą być błędne lub nieaktualne, dlatego zawsze sprawdź podaną stronę. Nie wpisuj swojego imienia ani innych danych osobowych.",
    "q_label": "Twoje pytanie",
    "q_hint": "Pisz w jednym z pięciu języków, maksymalnie {max} znaków. Odpowiedź będzie w języku pytania.",
    "ask": "Zapytaj",
    "examples": "Przykładowe pytania",
    "examples_hint": "Wybierz jedno, aby wstawić je do pola pytania.",
    "starting": "Asystent się uruchamia. Może to potrwać minutę lub dwie.",
    "ready": "Asystent jest gotowy.",
    "searching": "Szukam na oficjalnych stronach i piszę odpowiedź. Może to potrwać do minuty.",
    "answer_heading": "Odpowiedź",
    "refused_heading": "Nie znaleziono odpowiedzi",
    "answer_lang": "Język odpowiedzi: {lang}",
    "health_heading": "Pomoc medyczna",
    "sources_heading": "Źródła",
    "sources_intro": "Odpowiedź korzysta z kopii tych stron z podanego dnia. Otwórz stronę, aby sprawdzić, czy coś się zmieniło.",
    "searched_intro": "Ta odpowiedź nie pokazuje, z której strony pochodzi każde zdanie. Została napisana na podstawie tych stron, więc je sprawdź.",
    "source_n": "Źródło {n}",
    "updated": "Ostatnia aktualizacja strony",
    "copied": "Data kopii",
    "open": "Otwórz oficjalną stronę",
    "new_tab": "(otwiera się w nowej karcie)",
    "official_sites": "Oficjalne strony",
    "help": "Jeśli potrzebujesz pilnej pomocy medycznej, zadzwoń pod numer 111. W nagłym wypadku dzwoń pod 999.",
    "licence": "Contains public sector information licensed under the Open Government Licence v3.0.",
    "err_empty": "Wpisz pytanie.",
    "err_too_long": "Pytanie jest za długie. Możesz użyć najwyżej {max} znaków.",
    "err_loading": "Asystent jeszcze się uruchamia. Spróbuj ponownie za minutę.",
    "err_error": "Nie udało się uruchomić asystenta. Powiadom osobę prowadzącą badanie.",
    "err_server": "Coś poszło nie tak. Spróbuj ponownie.",
    "err_network": "Nie można połączyć się z asystentem. Sprawdź, czy działa, i spróbuj ponownie.",
    "err_bad_request": "Coś poszło nie tak. Spróbuj ponownie.",
    "examples_list": [
      "Jak zarejestrować się u lekarza rodzinnego?",
      "Jak często wynajmujący może podnosić czynsz?",
      "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?"
    ]
  },
  "ro": {
    "name": "Română",
    "page_title": "Întrebări despre sănătate, locuință și școală",
    "skip": "Salt la întrebare",
    "lang_label": "Limba",
    "title": "Întrebări despre sănătate, locuință și școală în Anglia",
    "intro": "Puneți o întrebare în engleză, poloneză, română, arabă sau urdu. Răspunsurile provin doar din paginile oficiale GOV.UK și NHS.uk și includ un link către fiecare pagină.",
    "notice_title": "Prototip de cercetare.",
    "notice": "Acesta este un instrument de cercetare, nu un sfat oficial. Răspunsurile pot fi greșite sau depășite, așa că verificați întotdeauna pagina indicată. Nu scrieți numele dumneavoastră sau alte date personale.",
    "q_label": "Întrebarea dumneavoastră",
    "q_hint": "Scrieți în oricare dintre cele cinci limbi, cel mult {max} de caractere. Răspunsul va fi în limba întrebării.",
    "ask": "Întrebați",
    "examples": "Exemple de întrebări",
    "examples_hint": "Alegeți una pentru a o pune în caseta de întrebare.",
    "starting": "Asistentul pornește. Poate dura un minut sau două.",
    "ready": "Asistentul este pregătit.",
    "searching": "Caut în paginile oficiale și scriu un răspuns. Poate dura până la un minut.",
    "answer_heading": "Răspuns",
    "refused_heading": "Nu am găsit un răspuns",
    "answer_lang": "Limba răspunsului: {lang}",
    "health_heading": "Ajutor medical",
    "sources_heading": "Surse",
    "sources_intro": "Răspunsul folosește copii ale acestor pagini, făcute la data indicată. Deschideți pagina pentru a verifica dacă s-a schimbat ceva.",
    "searched_intro": "Acest răspuns nu arată din ce pagină provine fiecare propoziție. A fost scris pe baza acestor pagini, așa că verificați-le.",
    "source_n": "Sursa {n}",
    "updated": "Pagină actualizată ultima dată",
    "copied": "Copie făcută la",
    "open": "Deschideți pagina oficială",
    "new_tab": "(se deschide într-o filă nouă)",
    "official_sites": "Site-uri oficiale",
    "help": "Dacă aveți nevoie urgent de ajutor medical, sunați la 111. În caz de urgență, sunați la 999.",
    "licence": "Contains public sector information licensed under the Open Government Licence v3.0.",
    "err_empty": "Scrieți o întrebare.",
    "err_too_long": "Întrebarea este prea lungă. Folosiți cel mult {max} de caractere.",
    "err_loading": "Asistentul încă pornește. Încercați din nou peste un minut.",
    "err_error": "Asistentul nu a putut porni. Spuneți-i cercetătorului.",
    "err_server": "A apărut o eroare. Încercați din nou.",
    "err_network": "Asistentul nu poate fi contactat. Verificați dacă rulează, apoi încercați din nou.",
    "err_bad_request": "A apărut o eroare. Încercați din nou.",
    "examples_list": [
      "Cum mă înregistrez la un medic de familie?",
      "Cât de des poate proprietarul să crească chiria?",
      "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?"
    ]
  },
  "ar": {
    "name": "العربية",
    "page_title": "أسئلة عن الصحة والسكن والمدارس",
    "skip": "انتقل إلى سؤالك",
    "lang_label": "اللغة",
    "title": "أسئلة عن الصحة والسكن والمدارس في إنجلترا",
    "intro": "اطرح سؤالك بالإنجليزية أو البولندية أو الرومانية أو العربية أو الأردية. تأتي الإجابات من صفحات GOV.UK وNHS.uk الرسمية فقط، مع رابط لكل صفحة.",
    "notice_title": "نموذج بحثي أولي.",
    "notice": "هذه أداة بحثية وليست نصيحة رسمية. قد تكون الإجابات خاطئة أو قديمة، لذا تحقق دائمًا من الصفحة المرفقة. لا تكتب اسمك أو أي معلومات شخصية أخرى.",
    "q_label": "سؤالك",
    "q_hint": "اكتب بأي لغة من اللغات الخمس، بما لا يزيد عن {max} حرف. ستكون الإجابة بلغة سؤالك.",
    "ask": "اسأل",
    "examples": "أمثلة على الأسئلة",
    "examples_hint": "اختر سؤالًا لوضعه في خانة السؤال.",
    "starting": "يجري تشغيل المساعد. قد يستغرق ذلك دقيقة أو دقيقتين.",
    "ready": "المساعد جاهز.",
    "searching": "يجري البحث في الصفحات الرسمية وكتابة الإجابة. قد يستغرق ذلك دقيقة.",
    "answer_heading": "الإجابة",
    "refused_heading": "لم يتم العثور على إجابة",
    "answer_lang": "لغة الإجابة: {lang}",
    "health_heading": "مساعدة طبية",
    "sources_heading": "المصادر",
    "sources_intro": "تعتمد الإجابة على نسخ من هذه الصفحات أُخذت في التاريخ المذكور. افتح الصفحة للتحقق من وجود أي تغييرات.",
    "searched_intro": "لا تُبيّن هذه الإجابة الصفحة التي جاءت منها كل جملة. كُتبت الإجابة اعتمادًا على هذه الصفحات، لذا راجعها.",
    "source_n": "المصدر {n}",
    "updated": "آخر تحديث للصفحة",
    "copied": "تاريخ النسخ",
    "open": "افتح الصفحة الرسمية",
    "new_tab": "(تُفتح في علامة تبويب جديدة)",
    "official_sites": "المواقع الرسمية",
    "help": "إذا كنت بحاجة إلى مساعدة طبية الآن، اتصل بالرقم 111. في حالات الطوارئ، اتصل بالرقم 999.",
    "licence": "Contains public sector information licensed under the Open Government Licence v3.0.",
    "err_empty": "اكتب سؤالًا.",
    "err_too_long": "سؤالك طويل جدًا. استخدم {max} حرف كحد أقصى.",
    "err_loading": "لا يزال المساعد قيد التشغيل. حاول مرة أخرى بعد دقيقة.",
    "err_error": "تعذّر تشغيل المساعد. أخبر الباحث.",
    "err_server": "حدث خطأ. حاول مرة أخرى.",
    "err_network": "تعذّر الاتصال بالمساعد. تأكد من أنه يعمل، ثم حاول مرة أخرى.",
    "err_bad_request": "حدث خطأ. حاول مرة أخرى.",
    "examples_list": [
      "كيف أسجل لدى طبيب عام؟",
      "كم مرة يمكن للمالك أن يرفع الإيجار؟",
      "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟"
    ]
  },
  "ur": {
    "name": "اردو",
    "page_title": "صحت، رہائش اور اسکول کے بارے میں سوالات",
    "skip": "اپنے سوال پر جائیں",
    "lang_label": "زبان",
    "title": "انگلینڈ میں صحت، رہائش اور اسکول کے بارے میں سوالات",
    "intro": "انگریزی، پولش، رومانین، عربی یا اردو میں سوال پوچھیں۔ جوابات صرف سرکاری GOV.UK اور NHS.uk صفحات سے لیے جاتے ہیں، اور ہر صفحے کا لنک دیا جاتا ہے۔",
    "notice_title": "تحقیقی نمونہ۔",
    "notice": "یہ ایک تحقیقی ٹول ہے، سرکاری مشورہ نہیں۔ جوابات غلط یا پرانے ہو سکتے ہیں، اس لیے ہمیشہ دیا گیا صفحہ دیکھیں۔ اپنا نام یا کوئی اور ذاتی معلومات نہ لکھیں۔",
    "q_label": "آپ کا سوال",
    "q_hint": "پانچ میں سے کسی بھی زبان میں لکھیں، زیادہ سے زیادہ {max} حروف۔ جواب آپ کے سوال کی زبان میں ہوگا۔",
    "ask": "پوچھیں",
    "examples": "سوالات کی مثالیں",
    "examples_hint": "سوال کے خانے میں ڈالنے کے لیے کوئی ایک منتخب کریں۔",
    "starting": "اسسٹنٹ شروع ہو رہا ہے۔ اس میں ایک یا دو منٹ لگ سکتے ہیں۔",
    "ready": "اسسٹنٹ تیار ہے۔",
    "searching": "سرکاری صفحات میں تلاش کی جا رہی ہے اور جواب لکھا جا رہا ہے۔ اس میں ایک منٹ تک لگ سکتا ہے۔",
    "answer_heading": "جواب",
    "refused_heading": "کوئی جواب نہیں ملا",
    "answer_lang": "جواب کی زبان: {lang}",
    "health_heading": "طبی مدد",
    "sources_heading": "ذرائع",
    "sources_intro": "جواب ان صفحات کی ان نقول پر مبنی ہے جو دی گئی تاریخ کو لی گئی تھیں۔ تبدیلیاں دیکھنے کے لیے صفحہ کھولیں۔",
    "searched_intro": "یہ جواب نہیں بتاتا کہ ہر جملہ کس صفحے سے لیا گیا ہے۔ یہ جواب ان صفحات کی مدد سے لکھا گیا ہے، اس لیے انہیں دیکھیں۔",
    "source_n": "ذریعہ {n}",
    "updated": "صفحہ آخری بار اپ ڈیٹ ہوا",
    "copied": "نقل کی تاریخ",
    "open": "سرکاری صفحہ کھولیں",
    "new_tab": "(نئے ٹیب میں کھلتا ہے)",
    "official_sites": "سرکاری ویب سائٹس",
    "help": "اگر آپ کو ابھی طبی مدد کی ضرورت ہے تو 111 پر کال کریں۔ ہنگامی صورت میں 999 پر کال کریں۔",
    "licence": "Contains public sector information licensed under the Open Government Licence v3.0.",
    "err_empty": "سوال لکھیں۔",
    "err_too_long": "آپ کا سوال بہت لمبا ہے۔ زیادہ سے زیادہ {max} حروف استعمال کریں۔",
    "err_loading": "اسسٹنٹ ابھی شروع ہو رہا ہے۔ ایک منٹ بعد دوبارہ کوشش کریں۔",
    "err_error": "اسسٹنٹ شروع نہیں ہو سکا۔ محقق کو بتائیں۔",
    "err_server": "کچھ غلط ہو گیا۔ دوبارہ کوشش کریں۔",
    "err_network": "اسسٹنٹ سے رابطہ نہیں ہو سکا۔ دیکھیں کہ یہ چل رہا ہے، پھر دوبارہ کوشش کریں۔",
    "err_bad_request": "کچھ غلط ہو گیا۔ دوبارہ کوشش کریں۔",
    "examples_list": [
      "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟",
      "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟",
      "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"
    ]
  }
}


In [ ]:
%%writefile tests/test_app.py
"""Tests for the week 8 web server (src/app.py) with a stub assistant, so no model is loaded.

Run:  .venv/bin/python -m unittest discover -s tests -v
"""
import contextlib
import io
import json
import pathlib
import sys
import threading
import unittest
import urllib.error
import urllib.request

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1] / "src"))
import app  # noqa: E402

NOTE = "If you need medical help now, call 111. In an emergency, call 999."
SOURCE = {"n": 1, "title": "Register with a GP surgery", "heading": "How to register", "source": "NHS.uk",
          "url": "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
          "public_updated_at": "2025-07-28", "fetched_at": "2026-10-28T10:08:02+00:00", "chunk": 7}


class StubAssistant:
    def __init__(self):
        self.questions = []

    def ask(self, question):
        self.questions.append(question)
        if "refuse" in question:
            return {"lang": "en", "refused": True, "reason": "low relevance", "health_note": None,
                    "answer": "I could not find an answer.", "sources": [], "raw": ""}
        if "boom" in question:
            raise RuntimeError("model failed")
        return {"lang": "pl", "refused": False, "reason": None, "raw": "secret raw output",
                "answer": f"Zarejestruj się online [1].\n\n{NOTE}", "health_note": NOTE,
                "sources": [SOURCE], "passages": [{"text": "full passage"}]}


class ServerTest(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.stub = StubAssistant()
        cls.backend = app.Backend(lambda: cls.stub).start()
        cls.backend.loaded.wait(5)
        cls.server = app.make_server(cls.backend, port=0)
        cls.base = f"http://127.0.0.1:{cls.server.server_address[1]}"
        threading.Thread(target=cls.server.serve_forever, daemon=True).start()

    @classmethod
    def tearDownClass(cls):
        cls.server.shutdown()
        cls.server.server_close()

    def request(self, path, body=None, raw=None):
        data = raw if raw is not None else (None if body is None else json.dumps(body).encode())
        req = urllib.request.Request(self.base + path, data=data, headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=5) as r:
                return r.status, dict(r.headers), r.read()
        except urllib.error.HTTPError as e:
            return e.code, dict(e.headers), e.read()

    def test_page_and_static_files(self):
        status, headers, body = self.request("/")
        self.assertEqual(status, 200)
        self.assertIn("text/html", headers["Content-Type"])
        self.assertIn(b'id="question"', body)
        self.assertIn("default-src 'self'", headers["Content-Security-Policy"])
        for name, ctype in (("app.js", "javascript"), ("style.css", "text/css"), ("i18n.json", "json")):
            status, headers, _ = self.request(f"/static/{name}")
            self.assertEqual(status, 200, name)
            self.assertIn(ctype, headers["Content-Type"])

    def test_no_files_outside_web_folder(self):
        for path in ("/static/../src/app.py", "/static/%2e%2e/README.md", "/static/", "/README.md",
                     "/static/missing.css", "/data/processed/chunks.csv"):
            self.assertEqual(self.request(path)[0], 404, path)

    def test_status(self):
        status, _, body = self.request("/api/status")
        self.assertEqual(status, 200)
        self.assertEqual(json.loads(body), {"status": "ready", "max_chars": app.MAX_CHARS})

    def test_answer_shape(self):
        status, headers, body = self.request("/api/ask", {"question": "  Jak zarejestrować\nsię u lekarza?  "})
        self.assertEqual(status, 200)
        self.assertEqual(headers["Cache-Control"], "no-store")
        res = json.loads(body)
        self.assertEqual(self.stub.questions[-1], "Jak zarejestrować się u lekarza?")  # whitespace collapsed
        self.assertEqual(res["answer"], "Zarejestruj się online [1].")              # note split off
        self.assertEqual(res["health_note"], NOTE)
        self.assertEqual(res["lang"], "pl")
        self.assertFalse(res["refused"])
        self.assertEqual(res["sources"], [{k: SOURCE[k] for k in app.SOURCE_KEYS}])
        for hidden in ("raw", "passages", "reason"):  # internals stay on the server
            self.assertNotIn(hidden, res)

    def test_refusal(self):
        res = json.loads(self.request("/api/ask", {"question": "please refuse"})[2])
        self.assertTrue(res["refused"])
        self.assertEqual(res["sources"], [])
        self.assertIsNone(res["health_note"])

    def test_bad_requests(self):
        cases = [({"question": "   "}, "empty"), ({"question": "x" * (app.MAX_CHARS + 1)}, "too_long"),
                 ({"q": "hi"}, "bad_request"), ({"question": 5}, "bad_request"), (["hi"], "bad_request")]
        for body, code in cases:
            status, _, out = self.request("/api/ask", body)
            self.assertEqual((status, json.loads(out)["error"]), (400, code), body)
        status, _, out = self.request("/api/ask", raw=b"not json")
        self.assertEqual((status, json.loads(out)["error"]), (400, "bad_request"))
        status, _, out = self.request("/api/ask", raw=b"x" * (app.MAX_BODY + 1))
        self.assertEqual((status, json.loads(out)["error"]), (400, "too_long"))
        self.assertEqual(self.request("/api/other", {"question": "hi"})[0], 404)

    def test_server_error_hides_details(self):
        with contextlib.redirect_stderr(io.StringIO()):  # the server prints the expected traceback
            status, _, body = self.request("/api/ask", {"question": "boom"})
        self.assertEqual((status, json.loads(body)), (500, {"error": "server"}))


class LoadingTest(unittest.TestCase):
    def test_not_ready_and_failed_load(self):
        gate = threading.Event()
        loading = app.Backend(lambda: gate.wait(5) and StubAssistant()).start()
        self.assertEqual(loading.status(), "loading")
        gate.set()
        loading.loaded.wait(5)
        self.assertEqual(loading.status(), "ready")

        def fail():
            raise OSError("model files missing")
        broken = app.Backend(fail)
        with contextlib.redirect_stderr(io.StringIO()):  # the traceback is expected
            broken._load()  # in this thread, to keep the test simple
        self.assertEqual(broken.status(), "error")
        self.assertIn("model files missing", broken.error)

    def test_ask_while_loading_returns_503(self):
        gate = threading.Event()
        backend = app.Backend(lambda: gate.wait(5) and StubAssistant()).start()
        server = app.make_server(backend, port=0)
        threading.Thread(target=server.serve_forever, daemon=True).start()
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{server.server_address[1]}/api/ask",
                                         data=json.dumps({"question": "hi"}).encode())
            with self.assertRaises(urllib.error.HTTPError) as ctx:
                urllib.request.urlopen(req, timeout=5)
            self.assertEqual(ctx.exception.code, 503)
            self.assertEqual(json.loads(ctx.exception.read())["error"], "loading")
        finally:
            gate.set()
            server.shutdown()
            server.server_close()


class PublicTest(unittest.TestCase):
    def test_note_only_removed_from_the_end(self):
        res = app.public({"lang": "en", "refused": False, "answer": "Call 111 [1].", "health_note": "Call 111.",
                          "sources": []})
        self.assertEqual(res["answer"], "Call 111 [1].")

    def test_uncited_answer_still_links_to_pages(self):
        passages = [{**SOURCE, "n": 99, "text": "passage"}, {**SOURCE, "title": "Second page", "text": "p2"}]
        res = app.public({"lang": "en", "refused": False, "answer": "No markers here.", "health_note": None,
                          "sources": [], "passages": passages})
        self.assertFalse(res["cited"])
        self.assertEqual([s["title"] for s in res["sources"]], [SOURCE["title"], "Second page"])
        self.assertTrue(all(s["n"] is None and "text" not in s for s in res["sources"]))

    def test_refusal_shows_no_pages(self):
        res = app.public({"lang": "en", "refused": True, "answer": "No.", "sources": [],
                          "passages": [{**SOURCE, "text": "p"}]})
        self.assertEqual(res["sources"], [])


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile tests/test_web.py
"""Static checks on the week 8 page (T35): translations complete, labels present, colour contrast.

These are the WCAG 2.2 AA checks that can be automated without a browser. The manual checks (keyboard,
screen reader, zoom, right-to-left reading) are in docs/accessibility.md.
"""
import ast
import csv
import json
import pathlib
import re
import unittest
from html.parser import HTMLParser

ROOT = pathlib.Path(__file__).resolve().parents[1]
WEB = ROOT / "web"
I18N = json.loads((WEB / "i18n.json").read_text(encoding="utf-8"))
LANGS = I18N["order"]
HTML = (WEB / "index.html").read_text(encoding="utf-8")
CSS = (WEB / "style.css").read_text(encoding="utf-8")
JS = (WEB / "app.js").read_text(encoding="utf-8")


def assistant_constant(name):
    """Read a dict constant from src/assistant.py without importing it (that would load torch)."""
    tree = ast.parse((ROOT / "src/assistant.py").read_text(encoding="utf-8"))
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(getattr(t, "id", None) == name for t in node.targets):
            return ast.literal_eval(node.value)
    raise KeyError(name)


# ---------- contrast (WCAG 2.x relative luminance) ----------
def luminance(hex_colour):
    h = hex_colour.lstrip("#")
    channels = [int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]
    lin = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in channels]
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2]


def contrast(a, b):
    la, lb = sorted((luminance(a), luminance(b)), reverse=True)
    return (la + 0.05) / (lb + 0.05)


TOKENS = dict(re.findall(r"--([\w-]+):\s*(#[0-9a-fA-F]{6})\s*;", CSS.split("}", 1)[0]))

# (foreground, background, where it is used). Text needs 4.5:1 (1.4.3).
TEXT_PAIRS = [
    ("ink", "bg", "body text"), ("ink", "surface", "source cards, footer"),
    ("muted", "bg", "hints, status, answer language"), ("muted", "surface", "card labels, dates, licence"),
    ("link", "bg", "links, example buttons"), ("link", "surface", "links on source cards"),
    ("link-visited", "bg", "visited links"), ("link-visited", "surface", "visited links on cards"),
    ("brand-ink", "brand", "header text, Ask button"), ("ink", "notice-bg", "research notice"),
    ("ink", "health-bg", "NHS 111/999 note"), ("error", "bg", "error message"),
    ("ink", "focus-halo", "skip link"),
]
# Borders of controls and focus indicators need 3:1 against what is next to them (1.4.11, 2.4.13 guidance).
UI_PAIRS = [
    ("ink", "bg", "text box and select border"), ("line", "bg", "example button border"),
    ("error", "bg", "invalid text box border"), ("focus-ring", "bg", "focus ring on the page"),
    ("focus-ring", "surface", "focus ring on cards and footer"), ("focus-halo", "brand", "focus halo on button and header"),
    ("brand", "bg", "Ask button against the page"),
]


class ContrastTest(unittest.TestCase):
    def test_tokens_found(self):
        for fg, bg, _ in TEXT_PAIRS + UI_PAIRS:
            self.assertIn(fg, TOKENS)
            self.assertIn(bg, TOKENS)

    def test_text_contrast(self):
        for fg, bg, where in TEXT_PAIRS:
            ratio = contrast(TOKENS[fg], TOKENS[bg])
            self.assertGreaterEqual(ratio, 4.5, f"{where}: --{fg} on --{bg} is {ratio:.2f}:1")

    def test_non_text_contrast(self):
        for fg, bg, where in UI_PAIRS:
            ratio = contrast(TOKENS[fg], TOKENS[bg])
            self.assertGreaterEqual(ratio, 3.0, f"{where}: --{fg} on --{bg} is {ratio:.2f}:1")


# ---------- translations ----------
class TranslationTest(unittest.TestCase):
    def test_every_language_has_every_key(self):
        keys = set(I18N["en"])
        for lang in LANGS:
            self.assertEqual(set(I18N[lang]), keys, lang)
            for key, value in I18N[lang].items():
                self.assertTrue(value, f"{lang}.{key} is empty")

    def test_placeholders_match_english(self):
        for key, en in I18N["en"].items():
            if isinstance(en, str):
                want = set(re.findall(r"\{\w+\}", en))
                for lang in LANGS:
                    self.assertEqual(set(re.findall(r"\{\w+\}", I18N[lang][key])), want, f"{lang}.{key}")

    def test_keys_used_by_page_exist(self):
        used = set(re.findall(r'data-i18n="([\w]+)"', HTML))
        used |= set(re.findall(r'\bt\("(\w+)"', JS))
        used |= {f"err_{c}" for c in ("empty", "too_long", "loading", "error", "server", "network", "bad_request")}
        missing = used - set(I18N["en"])
        self.assertFalse(missing, f"keys used but not translated: {missing}")

    def test_health_note_matches_assistant(self):
        note = assistant_constant("HEALTH_NOTE")
        for lang in LANGS:
            self.assertEqual(I18N[lang]["help"], note[lang], lang)

    def test_examples_are_test_set_questions(self):
        with open(ROOT / "eval/queries.csv", encoding="utf-8") as f:
            rows = {(r["qid"], r["lang"]): r["question"] for r in csv.DictReader(f)}
        for lang in LANGS:
            want = [rows[(qid, lang)] for qid in ("M01", "H02", "S06")]
            self.assertEqual(I18N[lang]["examples_list"], want, lang)

    def test_rtl_languages(self):
        self.assertEqual(sorted(I18N["rtl"]), ["ar", "ur"])
        self.assertEqual(sorted(LANGS), sorted(assistant_constant("REFUSAL")))


# ---------- page structure ----------
class PageParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.tags = []

    def handle_starttag(self, tag, attrs):
        self.tags.append((tag, dict(attrs)))


class PageTest(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        p = PageParser()
        p.feed(HTML)
        cls.tags = p.tags

    def find(self, tag):
        return [a for t, a in self.tags if t == tag]

    def test_language_and_direction_declared(self):
        html = self.find("html")[0]
        self.assertEqual((html.get("lang"), html.get("dir")), ("en", "ltr"))
        self.assertIn("root.lang = lang", JS)
        self.assertIn("root.dir = dirOf(lang)", JS)

    def test_zoom_not_blocked(self):
        viewport = [a for a in self.find("meta") if a.get("name") == "viewport"][0]["content"]
        self.assertNotIn("user-scalable=no", viewport)
        self.assertNotIn("maximum-scale", viewport)

    def test_form_controls_have_labels(self):
        labelled = {a["for"] for a in self.find("label")}
        for tag in ("textarea", "select", "input"):
            for a in self.find(tag):
                self.assertIn(a.get("id"), labelled, f"<{tag} id={a.get('id')}> has no <label>")

    def test_one_h1_and_landmarks(self):
        names = [t for t, _ in self.tags]
        self.assertEqual(names.count("h1"), 1)
        for landmark in ("header", "main", "footer"):
            self.assertIn(landmark, names)

    def test_live_region_for_status(self):
        status = [a for t, a in self.tags if a.get("id") == "status"][0]
        self.assertEqual(status.get("role"), "status")

    def test_new_tab_links_say_so(self):
        for a in self.find("a"):
            if a.get("target") == "_blank":
                self.assertIn("noopener", a.get("rel", ""))
        self.assertEqual(HTML.count('target="_blank"'), HTML.count('data-i18n="new_tab"'))
        self.assertIn('t("new_tab")', JS)

    def test_no_inline_script_or_style(self):
        """The Content-Security-Policy allows only files from this server, so inline code would be blocked."""
        for a in self.find("script"):
            self.assertIn("src", a)
        self.assertFalse(self.find("style"))
        self.assertFalse([a for _, a in self.tags if "style" in a or any(k.startswith("on") for k in a)])

    def test_model_text_never_parsed_as_html(self):
        found = re.findall(r"\.(?:inner|outer)HTML\s*\+?=|insertAdjacentHTML\s*\(|document\.write\s*\(", JS)
        self.assertEqual(found, [])

    def test_targets_at_least_24px(self):
        """2.5.8 Target Size (Minimum): buttons and the select are 2.75rem = 44px tall."""
        for selector in (".button", ".example", "select"):
            block = re.search(re.escape(selector) + r"\s*\{([^}]*)\}", CSS).group(1)
            self.assertIn("min-height: 2.75rem", block, selector)


if __name__ == "__main__":
    unittest.main()


In [ ]:
!{sys.executable} src/ingest.py

In [ ]:
!{sys.executable} src/chunk.py

In [ ]:
!{sys.executable} eval/build_queries.py

In [ ]:
import json, pathlib
import pandas as pd
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
pd.set_option('display.max_colwidth', 120)
print('GPU:', torch.cuda.is_available())

## What week 8 adds

```
browser: web/index.html + app.js + style.css + i18n.json  (5 interface languages, right-to-left for ar/ur)
   |  POST /api/ask {"question": ...}
src/app.py  (Python http.server, 127.0.0.1 only, no outside requests, questions not logged)
   |  Assistant.ask - the week 6 assistant, unchanged (direct mode, threshold 0.0052, yes/no check)
   v
answer + answer language + refusal flag + NHS 111/999 note + source cards (title, section, link, 2 dates)
```

* **Answer language follows the question**; the picker sets the interface language.
* **`[n]` markers become links** to numbered source cards with *page last updated* and *copy taken on* dates.
* **Every answer links to official pages**: an answer the model wrote without `[n]` markers lists the pages it was written from (section 3).
* **Model text is inserted as text, never as HTML**, so a model output cannot add links or markup.

## 1. Tests (no models needed)

`tests/test_app.py` runs the real server with a stub assistant: page and static files, path traversal, status, answer shape, refusal, bad requests, server errors, loading state, uncited answers. `tests/test_web.py` checks translations, labels, page structure and colour contrast.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT,
                   capture_output=True, text=True)
lines = [l for l in r.stderr.splitlines() if l.endswith(('ok', 'FAIL', 'ERROR')) or l.startswith(('Ran', 'OK', 'FAILED'))]
print('\n'.join(lines))

## 2. Live answers and timing on the GPU (T35a)

The week 6 assistant, unchanged. Three topics (M01 GP registration, H02 rent increases, S06 free school meals) in all five languages, plus two questions the sources do not cover. Week 6 on the laptop: about 19 s per answer that was not refused.

In [ ]:
import time
from assistant import Assistant
t0 = time.time(); bot = Assistant(); print(f'models loaded in {time.time() - t0:.0f} s')

In [ ]:
q = pd.read_csv('eval/queries.csv')
jobs = [(r.qid, r.lang, r.question) for r in q[q.qid.isin(['M01', 'H02', 'S06'])].itertuples()]
jobs += [('X1', 'en', 'How do I renew my UK passport?'), ('X2', 'ur', 'میں برطانوی پاسپورٹ کی تجدید کیسے کروں؟')]
rows = []
for qid, lang, question in jobs:
    t0 = time.time(); a = bot.ask(question)
    rows.append({'qid': qid, 'lang': lang, 'detected': a['lang'], 'seconds': round(time.time() - t0, 1),
                 'refused': a['refused'], 'sources': len(a['sources']), 'answer': a['answer'][:160]})
    print(qid, lang, rows[-1]['seconds'], 's', 'refused' if a['refused'] else '')
live = pd.DataFrame(rows)
print('median seconds, answered:', live[~live.refused].seconds.median(),
      '| refused:', live[live.refused].seconds.median())
live

## 3. Open the web app

The server runs inside this notebook on port 8000 and reuses the models loaded above. Colab shows a link that only you can open (it goes through your Google login, not the open internet). Try the language picker and an Arabic or Urdu question. Stop it with the last cell.

In [ ]:
import threading, app
backend = app.Backend(lambda: bot).start()
server = app.make_server(backend, host='127.0.0.1', port=8000)
threading.Thread(target=server.serve_forever, daemon=True).start()
from google.colab import output
output.serve_kernel_port_as_window(8000, path='/?lang=ar')

In [ ]:
server.shutdown(); server.server_close(); print('stopped')

## 4. Colour contrast (WCAG 2.2 1.4.3 and 1.4.11)

Every colour pair used in `web/style.css`. Text needs 4.5:1; borders of controls and focus indicators 3:1.

In [ ]:
sys.path.insert(0, str(ROOT / 'tests'))
import test_web as tw
rows = [{'kind': kind, 'foreground': fg, 'background': bg, 'used for': where,
         'ratio': round(tw.contrast(tw.TOKENS[fg], tw.TOKENS[bg]), 2), 'needs': need}
        for kind, pairs, need in (('text', tw.TEXT_PAIRS, 4.5), ('non-text', tw.UI_PAIRS, 3.0))
        for fg, bg, where in pairs]
contrast = pd.DataFrame(rows)
contrast['pass'] = contrast.ratio >= contrast.needs
contrast

## Findings

1. **The page works in all five languages.** The answer block takes the answer's own language and direction, so an Arabic question in the English interface gets a right-to-left Arabic answer; English titles and links inside Arabic and Urdu pages stay left-to-right.
2. **Accessibility (automated):** 0 axe violations in 10 states, lowest text contrast 6.54:1, no horizontal scroll at 320 px. Screen readers and real phones are still to check.
3. **Uncited answers** (about 4%: 6/150 in week 6, 10/248 in week 7) now still link to official pages.
4. **Live answers on the laptop were too slow this week.** Another virtual machine was using about 9 of the 15 CPU cores; NLLB on the CPU took over 3 minutes per sentence (0.3 s on the Apple GPU, now used by the web app) and Qwen did not produce 10 tokens in 150 s. Week 6 on the same laptop: about 19 s per answer. The Colab notebook times live answers on a T4 GPU (T35a); time the study machine again before week 9 sessions.

Full write-up: `docs/week-08-report.md`; accessibility details: `docs/accessibility.md`.